In [1]:

# ============================================================
# REMOUNT GOOGLE DRIVE + VERIFY FRESH RESEARCH DATASET
# ============================================================

!fusermount -u /content/drive 2>/dev/null || true
!rm -rf /content/drive

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")
BASE = DRIVE / "Bitbrains_Fresh_Research"

# Find patchSk anywhere under MyDrive
matches = [
    p for p in DRIVE.rglob("*")
    if p.is_dir() and p.name.lower() == "patchsk"
]

print("\n" + "=" * 70)
print("CHECKING FRESH BITBRAINS RESEARCH FILES")
print("=" * 70)

print(f"{'✓' if BASE.exists() else '✗'} Fresh project folder")
print("  ", BASE)

if matches:
    DATA_ROOT = matches[0]
    print(f"{'✓' if DATA_ROOT.exists() else '✗'} Uploaded dataset folder")
    print("  ", DATA_ROOT)
    print("  Files:", sum(1 for p in DATA_ROOT.rglob("*") if p.is_file()))
else:
    DATA_ROOT = None
    print("✗ patchSk folder not visible in mounted MyDrive")

paths = {
    "Raw workspace": BASE / "data/raw",
    "Processed workspace": BASE / "data/processed",
    "Reports workspace": BASE / "reports",
}

for name, path in paths.items():
    print(f"{'✓' if path.exists() else '✗'} {name}")
    print("  ", path)

print("\n" + "=" * 70)

if DATA_ROOT is not None:
    print("✅ Dataset located. Ready for the fresh data audit.")
else:
    print("⚠️ Dataset folder is still missing from mounted Drive.")
    print("Check that you authorized the Google account containing patchSk.")


Mounted at /content/drive

CHECKING FRESH BITBRAINS RESEARCH FILES
✓ Fresh project folder
   /content/drive/MyDrive/Bitbrains_Fresh_Research
✓ Uploaded dataset folder
   /content/drive/MyDrive/patchSk
  Files: 1250
✗ Raw workspace
   /content/drive/MyDrive/Bitbrains_Fresh_Research/data/raw
✓ Processed workspace
   /content/drive/MyDrive/Bitbrains_Fresh_Research/data/processed
✓ Reports workspace
   /content/drive/MyDrive/Bitbrains_Fresh_Research/reports

✅ Dataset located. Ready for the fresh data audit.


In [ ]:

# ============================================================
# FRESH BITBRAINS — COMPLETE PREPROCESSING
# Raw data stays untouched. New outputs go to Bitbrains_Fresh_Research.
# ============================================================

import os, json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

DRIVE = Path("/content/drive/MyDrive")
RAW_ROOT = DRIVE / "patchSk"
PROJECT = DRIVE / "Bitbrains_Fresh_Research"
PROCESSED = PROJECT / "data/processed/bitbrains_5min"
MODEL_DATA = PROJECT / "data/model_data"
REPORTS = PROJECT / "reports"

for p in [PROCESSED, MODEL_DATA, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

# Model uses four core resources.
CORE = [
    "CPU usage [%]",
    "Memory usage [KB]",
    "Disk read throughput [KB/s]",
    "Disk write throughput [KB/s]",
]
ALL_RESOURCES = CORE + [
    "Network received throughput [KB/s]",
    "Network transmitted throughput [KB/s]",
]

LOOKBACK, HORIZON = 96, 12
MIN_DURATION_DAYS = 7
FREQ = "5min"

# Discover CSV files recursively; do not assume they are at the root.
raw_files = sorted(
    [p for p in RAW_ROOT.rglob("*.csv") if p.is_file()]
)

print("Raw CSV files found:", len(raw_files))
if not raw_files:
    raise FileNotFoundError(
        f"No CSV traces found under {RAW_ROOT}. "
        "Check whether patchSk contains another archive that needs extraction."
    )

# If there are duplicate basenames, use a stable unique ID per file.
records, errors = [], []
used_ids = set()

for i, path in enumerate(raw_files, start=1):
    try:
        # Bitbrains traces use semicolon-separated columns.
        df = pd.read_csv(path, sep=";", engine="python")
        df.columns = [str(c).strip().rstrip(";").strip() for c in df.columns]

        # Remove a possible unnamed trailing column.
        df = df.loc[:, ~df.columns.str.match(r"^Unnamed")]

        required = ["Timestamp [ms]"] + ALL_RESOURCES
        missing_cols = [c for c in required if c not in df.columns]
        if missing_cols:
            errors.append({
                "file": str(path.relative_to(RAW_ROOT)),
                "error": f"Missing columns: {missing_cols}",
            })
            continue

        # Previous forensic analysis found these timestamp values
        # behave as Unix seconds despite the source column name.
        ts = pd.to_numeric(df["Timestamp [ms]"], errors="coerce")
        timestamps = pd.to_datetime(ts, unit="s", errors="coerce")

        work = df[ALL_RESOURCES].apply(pd.to_numeric, errors="coerce")
        work.insert(0, "Timestamp", timestamps)
        work = work.dropna(subset=["Timestamp"])

        # Invalid negative resource values become missing.
        for col in ALL_RESOURCES:
            work.loc[work[col] < 0, col] = np.nan

        # Sort and aggregate duplicate timestamps.
        work = (
            work.sort_values("Timestamp")
                .groupby("Timestamp", as_index=False)[ALL_RESOURCES]
                .mean()
        )

        if len(work) < 2:
            continue

        duration_days = (
            work["Timestamp"].max() - work["Timestamp"].min()
        ).total_seconds() / 86400

        if duration_days < MIN_DURATION_DAYS:
            records.append({
                "source_file": str(path.relative_to(RAW_ROOT)),
                "status": "excluded_short_trace",
                "duration_days": duration_days,
            })
            continue

        # Five-minute aggregation; no interpolation or extrapolation.
        resampled = (
            work.set_index("Timestamp")[ALL_RESOURCES]
                .resample(FREQ)
                .mean()
        )

        missing_core = resampled[CORE].isna().any(axis=1)
        valid = ~missing_core

        # Count windows only within continuous valid segments.
        arr = valid.to_numpy(dtype=np.int8)
        padded = np.r_[0, arr, 0]
        changes = np.diff(padded)
        starts = np.where(changes == 1)[0]
        ends = np.where(changes == -1)[0]
        segment_lengths = ends - starts

        usable_windows = int(
            sum(max(0, n - LOOKBACK - HORIZON + 1)
                for n in segment_lengths)
        )

        # Stable ID: use numeric filename if possible and unique;
        # otherwise allocate the next unused integer.
        stem = path.stem
        candidate = int(stem) if stem.isdigit() else None
        if candidate is None or candidate in used_ids:
            candidate = max(used_ids, default=0) + 1
            while candidate in used_ids:
                candidate += 1
        used_ids.add(candidate)

        out = resampled[CORE].copy()
        out.insert(0, "Timestamp", out.index)
        out.insert(1, "VM_ID", candidate)
        out.to_csv(PROCESSED / f"{candidate}.csv", index=False)

        records.append({
            "vm_id": candidate,
            "source_file": str(path.relative_to(RAW_ROOT)),
            "status": "processed",
            "raw_rows": len(df),
            "unique_timestamps": len(work),
            "duration_days": duration_days,
            "resampled_points": len(out),
            "missing_core_points": int(missing_core.sum()),
            "missing_core_pct": round(100 * missing_core.mean(), 3),
            "usable_windows_before_split": usable_windows,
        })

        if i % 100 == 0 or i == len(raw_files):
            print(f"Processed {i}/{len(raw_files)} files")

    except Exception as e:
        errors.append({
            "file": str(path.relative_to(RAW_ROOT)),
            "error": repr(e),
        })

report = pd.DataFrame(records)
report.to_csv(REPORTS / "preprocessing_report.csv", index=False)
pd.DataFrame(errors).to_csv(REPORTS / "preprocessing_errors.csv", index=False)

print("\n" + "=" * 65)
print("PREPROCESSING SUMMARY")
print("=" * 65)
print("Input CSV files:", len(raw_files))
print("Processed VMs:", int((report["status"] == "processed").sum())
      if "status" in report else 0)
print("Excluded short traces:", int((report["status"] == "excluded_short_trace").sum())
      if "status" in report else 0)
print("Files with errors:", len(errors))
print("Processed output:", PROCESSED)
print("Report:", REPORTS / "preprocessing_report.csv")
print("Errors:", REPORTS / "preprocessing_errors.csv")


Raw CSV files found: 1250
Processed 100/1250 files
Processed 200/1250 files
Processed 300/1250 files
Processed 400/1250 files
Processed 500/1250 files
Processed 600/1250 files
Processed 700/1250 files
Processed 800/1250 files
Processed 900/1250 files
Processed 1000/1250 files
Processed 1100/1250 files
Processed 1200/1250 files
Processed 1250/1250 files

PREPROCESSING SUMMARY
Input CSV files: 1250
Processed VMs: 1217
Excluded short traces: 33
Files with errors: 0
Processed output: /content/drive/MyDrive/Bitbrains_Fresh_Research/data/processed/bitbrains_5min
Report: /content/drive/MyDrive/Bitbrains_Fresh_Research/reports/preprocessing_report.csv
Errors: /content/drive/MyDrive/Bitbrains_Fresh_Research/reports/preprocessing_errors.csv


In [ ]:

# ============================================================
# FRESH BITBRAINS — SPLITS + TRAIN-ONLY SCALER
# ============================================================

import json, pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler

PROJECT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
PROCESSED = PROJECT / "data/processed/bitbrains_5min"
SPLITS = PROJECT / "data/splits"
MODEL_DATA = PROJECT / "data/model_data"
REPORTS = PROJECT / "reports"

for p in [SPLITS, MODEL_DATA, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "CPU usage [%]",
    "Memory usage [KB]",
    "Disk read throughput [KB/s]",
    "Disk write throughput [KB/s]",
]
LOOKBACK, HORIZON = 96, 12
TRAIN_RATIO, VAL_RATIO = 0.70, 0.15

files = sorted(
    [p for p in PROCESSED.glob("*.csv") if p.stem.isdigit()],
    key=lambda p: int(p.stem)
)
if not files:
    raise FileNotFoundError("No processed VM CSV files found. Run Cell 1 first.")

split_rows = []
scaler = StandardScaler()
train_rows_total = 0
eligible_vms = 0

# First pass: create chronological split files and fit scaler on
# valid training rows only.
for idx, path in enumerate(files, start=1):
    df = pd.read_csv(path, parse_dates=["Timestamp"])
    if not set(FEATURES).issubset(df.columns):
        continue

    n = len(df)
    train_end = int(n * TRAIN_RATIO)
    val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

    # Keep temporal order; no random splitting.
    train = df.iloc[:train_end].copy()
    val = df.iloc[train_end:val_end].copy()
    test = df.iloc[val_end:].copy()

    # Fit scaler only on complete, finite training rows.
    train_values = train[FEATURES].to_numpy(dtype=np.float64)
    good = np.isfinite(train_values).all(axis=1)
    if good.sum() == 0:
        continue

    scaler.partial_fit(train_values[good])
    train_rows_total += int(good.sum())
    eligible_vms += 1

    for split_name, part in [
        ("train", train), ("val", val), ("test", test)
    ]:
        out = SPLITS / split_name / f"{path.stem}.csv"
        out.parent.mkdir(parents=True, exist_ok=True)
        part.to_csv(out, index=False)

    split_rows.append({
        "vm_id": int(path.stem),
        "total_rows": n,
        "train_rows": len(train),
        "val_rows": len(val),
        "test_rows": len(test),
        "train_valid_rows": int(good.sum()),
        "val_valid_rows": int(np.isfinite(val[FEATURES].to_numpy()).all(axis=1).sum()),
        "test_valid_rows": int(np.isfinite(test[FEATURES].to_numpy()).all(axis=1).sum()),
    })

if train_rows_total == 0:
    raise RuntimeError("No valid training rows available to fit scaler.")

with open(MODEL_DATA / "train_only_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

config = {
    "features": FEATURES,
    "lookback": LOOKBACK,
    "horizon": HORIZON,
    "frequency": "5min",
    "train_ratio": TRAIN_RATIO,
    "validation_ratio": VAL_RATIO,
    "test_ratio": 1 - TRAIN_RATIO - VAL_RATIO,
    "scaler_fit_rows": train_rows_total,
    "eligible_vms": eligible_vms,
    "scaler_fit_scope": "complete finite rows in each VM's training split only",
}
with open(MODEL_DATA / "dataset_config.json", "w") as f:
    json.dump(config, f, indent=2)

split_df = pd.DataFrame(split_rows)
split_df.to_csv(MODEL_DATA / "split_metadata.csv", index=False)

print("=" * 65)
print("SPLITTING AND SCALING COMPLETE")
print("=" * 65)
print("Processed VM files:", len(files))
print("VMs included in split/scaler:", eligible_vms)
print("Valid training rows used for scaler:", train_rows_total)
print("Train split:", SPLITS / "train")
print("Validation split:", SPLITS / "val")
print("Test split:", SPLITS / "test")
print("Scaler:", MODEL_DATA / "train_only_scaler.pkl")
print("Metadata:", MODEL_DATA / "split_metadata.csv")


SPLITTING AND SCALING COMPLETE
Processed VM files: 1217
VMs included in split/scaler: 1217
Valid training rows used for scaler: 7200654
Train split: /content/drive/MyDrive/Bitbrains_Fresh_Research/data/splits/train
Validation split: /content/drive/MyDrive/Bitbrains_Fresh_Research/data/splits/val
Test split: /content/drive/MyDrive/Bitbrains_Fresh_Research/data/splits/test
Scaler: /content/drive/MyDrive/Bitbrains_Fresh_Research/data/model_data/train_only_scaler.pkl
Metadata: /content/drive/MyDrive/Bitbrains_Fresh_Research/data/model_data/split_metadata.csv


In [ ]:

# ============================================================
# FRESH FORECASTING EXPERIMENT
# DLinear vs PatchTST vs PatchTST + RevIN
# Reuses existing splits and train-only scaler
# ============================================================

import os, json, pickle, time, copy, random, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -------------------- CONFIG --------------------

SEED = 42
LOOKBACK, HORIZON = 96, 12
BATCH_SIZE = 256
EPOCHS = 8
PATIENCE = 3
LR = 1e-3

# Same fixed window caps for every model; increase for final runs.
MAX_TRAIN_WINDOWS = 300_000
MAX_VAL_WINDOWS = 50_000
MAX_TEST_WINDOWS = 100_000

FEATURES = [
    "CPU usage [%]",
    "Memory usage [KB]",
    "Disk read throughput [KB/s]",
    "Disk write throughput [KB/s]",
]

PROJECT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
SPLITS = PROJECT / "data/splits"
MODEL_DATA = PROJECT / "data/model_data"
OUT = PROJECT / "results/forecast_comparison"
CKPTS = PROJECT / "models/forecast_comparison"

OUT.mkdir(parents=True, exist_ok=True)
CKPTS.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

assert (SPLITS / "train").exists(), "Missing train split. Run preprocessing/splitting first."
assert (SPLITS / "val").exists(), "Missing validation split."
assert (SPLITS / "test").exists(), "Missing test split."

with open(MODEL_DATA / "train_only_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# -------------------- LAZY VALID WINDOW DATASET --------------------
# Windows never cross VM boundaries, missing rows, or 5-min gaps.

class TraceWindows(Dataset):
    def __init__(self, split, max_windows=None, seed=42):
        self.arrays = []
        self.starts = []
        self.ends = []
        self.split = split

        files = sorted(
            (SPLITS / split).glob("*.csv"),
            key=lambda p: int(p.stem) if p.stem.isdigit() else p.stem
        )

        for path in files:
            df = pd.read_csv(path, parse_dates=["Timestamp"])
            if not set(FEATURES).issubset(df.columns):
                continue

            ts = pd.to_datetime(df["Timestamp"], errors="coerce")
            raw = df[FEATURES].to_numpy(dtype=np.float64)

            # Check timestamp interpretation and regular sampling.
            if ts.notna().sum() < LOOKBACK + HORIZON:
                continue
            if len(ts) > 1:
                med = ts.diff().dropna().dt.total_seconds().median()
                if not np.isfinite(med) or abs(med - 300) > 1:
                    continue

            values = scaler.transform(raw).astype(np.float32)
            valid = np.isfinite(values).all(axis=1) & ts.notna().to_numpy()

            # Segment boundaries ensure no window crosses a missing point.
            padded = np.r_[0, valid.astype(np.int8), 0]
            changes = np.diff(padded)
            seg_starts = np.where(changes == 1)[0]
            seg_ends = np.where(changes == -1)[0]

            for a, b in zip(seg_starts, seg_ends):
                # Require a continuous 5-minute timestamp grid.
                if b - a < LOOKBACK + HORIZON:
                    continue
                t = ts.iloc[a:b].astype("int64").to_numpy()
                if len(t) > 1 and not np.all(np.diff(t) == 300 * 10**9):
                    # Break this segment into smaller regular subsegments.
                    breaks = np.where(np.diff(t) != 300 * 10**9)[0] + 1
                    boundaries = np.r_[0, breaks, len(t)]
                    subsegments = [
                        (a + boundaries[j], a + boundaries[j+1])
                        for j in range(len(boundaries)-1)
                    ]
                else:
                    subsegments = [(a, b)]

                for s, e in subsegments:
                    nstarts = e - s - LOOKBACK - HORIZON + 1
                    if nstarts <= 0:
                        continue
                    arr_id = len(self.arrays)
                    self.arrays.append(values[s:e])
                    self.starts.extend([(arr_id, k) for k in range(nstarts)])

        if not self.starts:
            raise RuntimeError(
                f"No valid windows for split={split}. "
                "Check timestamp parsing and resampling frequency."
            )

        # Deterministic, reproducible cap shared by all models.
        if max_windows is not None and len(self.starts) > max_windows:
            rng = np.random.default_rng(seed)
            ids = np.sort(rng.choice(len(self.starts), max_windows, replace=False))
            self.starts = [self.starts[i] for i in ids]

        print(f"{split}: {len(self.starts):,} windows")

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        arr_id, start = self.starts[idx]
        arr = self.arrays[arr_id]
        x = arr[start:start + LOOKBACK]
        y = arr[start + LOOKBACK:start + LOOKBACK + HORIZON]
        return torch.from_numpy(x), torch.from_numpy(y)

train_ds = TraceWindows("train", MAX_TRAIN_WINDOWS, SEED)
val_ds   = TraceWindows("val", MAX_VAL_WINDOWS, SEED + 1)
test_ds  = TraceWindows("test", MAX_TEST_WINDOWS, SEED + 2)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

# -------------------- MODEL 1: DLINEAR --------------------

class DLinear(nn.Module):
    def __init__(self, seq_len=96, pred_len=12, channels=4, kernel=25):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.channels = channels
        self.moving_avg = nn.AvgPool1d(
            kernel_size=kernel, stride=1, padding=0
        )
        self.linear_trend = nn.ModuleList(
            [nn.Linear(seq_len, pred_len) for _ in range(channels)]
        )
        self.linear_seasonal = nn.ModuleList(
            [nn.Linear(seq_len, pred_len) for _ in range(channels)]
        )
        self.kernel = kernel

    def forward(self, x):
        # x: [B, L, C]
        xt = x.transpose(1, 2)
        pad = (self.kernel - 1) // 2
        padded = torch.nn.functional.pad(xt, (pad, pad), mode="replicate")
        trend = torch.nn.functional.avg_pool1d(
            padded, kernel_size=self.kernel, stride=1
        )
        seasonal = xt - trend

        outputs = []
        for c in range(self.channels):
            outputs.append(
                self.linear_trend[c](trend[:, c, :]) +
                self.linear_seasonal[c](seasonal[:, c, :])
            )
        return torch.stack(outputs, dim=-1)

# -------------------- PATCHTST --------------------

class PatchTST(nn.Module):
    def __init__(
        self, seq_len=96, pred_len=12, channels=4,
        patch_len=16, stride=8, d_model=128,
        n_heads=8, layers=3, ff_dim=256, dropout=0.1,
        revin=False
    ):
        super().__init__()
        self.channels = channels
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.patch_len = patch_len
        self.stride = stride
        self.revin = revin

        n_patches = (seq_len - patch_len) // stride + 1
        self.patch_proj = nn.Linear(patch_len, d_model)
        self.pos = nn.Parameter(torch.zeros(1, n_patches, d_model))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True, norm_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.head = nn.Linear(n_patches * d_model, pred_len)
        nn.init.trunc_normal_(self.pos, std=0.02)

    def forward(self, x):
        # Channel-independent PatchTST. RevIN stats come from input only.
        # x: [B, L, C]
        B, L, C = x.shape
        z = x.transpose(1, 2).reshape(B * C, L)

        if self.revin:
            mu = z.mean(dim=1, keepdim=True).detach()
            sd = torch.sqrt(z.var(dim=1, keepdim=True, unbiased=False) + 1e-5).detach()
            z = (z - mu) / sd

        patches = z.unfold(dimension=1, size=self.patch_len, step=self.stride)
        z = self.patch_proj(patches)
        z = z + self.pos[:, :z.size(1), :]
        z = self.encoder(z)
        z = self.head(z.flatten(1))

        if self.revin:
            z = z * sd + mu

        return z.reshape(B, C, self.pred_len).transpose(1, 2)

# -------------------- TRAIN / VALIDATE --------------------

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total_n = 0.0, 0

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(training):
            pred = model(x)
            loss = torch.nn.functional.huber_loss(pred, y, delta=1.0)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_n += x.size(0)

    return total_loss / max(total_n, 1)

def train_model(name, model):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=1
    )

    best_val = float("inf")
    best_state = None
    bad_epochs = 0
    history = []
    start_time = time.time()

    for epoch in range(1, EPOCHS + 1):
        tr = run_epoch(model, train_loader, optimizer)
        va = run_epoch(model, val_loader)
        scheduler.step(va)

        history.append({
            "model": name, "epoch": epoch,
            "train_huber": tr, "val_huber": va,
            "lr": optimizer.param_groups[0]["lr"]
        })

        print(
            f"{name} | Epoch {epoch}/{EPOCHS} | "
            f"train={tr:.6f} | val={va:.6f}"
        )

        if va < best_val:
            best_val = va
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
            torch.save(best_state, CKPTS / f"{name}_best.pt")
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("Early stopping.")
                break

    model.load_state_dict(best_state)
    pd.DataFrame(history).to_csv(OUT / f"{name}_history.csv", index=False)
    return model, best_val, time.time() - start_time

def evaluate(model, loader):
    model.eval()
    abs_sum = np.zeros(len(FEATURES))
    sq_sum = np.zeros(len(FEATURES))
    y_sum = np.zeros(len(FEATURES))
    y_sq_sum = np.zeros(len(FEATURES))
    n = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            pred = model(x).cpu().numpy()
            actual = y.numpy()
            err = pred - actual
            abs_sum += np.abs(err).sum(axis=(0, 1))
            sq_sum += (err ** 2).sum(axis=(0, 1))
            y_sum += actual.sum(axis=(0, 1))
            y_sq_sum += (actual ** 2).sum(axis=(0, 1))
            n += actual.shape[0] * actual.shape[1]

    mae = abs_sum / n
    rmse = np.sqrt(sq_sum / n)
    variance = y_sq_sum / n - (y_sum / n) ** 2
    r2 = 1 - (sq_sum / n) / np.maximum(variance, 1e-12)

    return mae, rmse, r2

# -------------------- RUN ALL THREE --------------------

models = {
    "DLinear": DLinear(),
    "PatchTST": PatchTST(revin=False),
    "PatchTST_RevIN": PatchTST(revin=True),
}

summary = []
per_resource = []

for name, model in models.items():
    print("\n" + "=" * 70)
    print("TRAINING:", name)
    print("=" * 70)

    model, best_val, seconds = train_model(name, model)
    mae, rmse, r2 = evaluate(model, test_loader)

    summary.append({
        "model": name,
        "parameters": sum(p.numel() for p in model.parameters()),
        "best_val_huber": best_val,
        "test_mae_mean_scaled": float(mae.mean()),
        "test_rmse_mean_scaled": float(rmse.mean()),
        "test_r2_mean": float(r2.mean()),
        "training_seconds": seconds,
        "test_windows": len(test_ds),
    })

    for i, feature in enumerate(FEATURES):
        per_resource.append({
            "model": name,
            "resource": feature,
            "MAE_scaled": mae[i],
            "RMSE_scaled": rmse[i],
            "R2": r2[i],
        })

    print("Test MAE (scaled):", dict(zip(FEATURES, mae.round(5))))
    print("Test RMSE (scaled):", dict(zip(FEATURES, rmse.round(5))))
    print("Test R²:", dict(zip(FEATURES, r2.round(5))))

summary_df = pd.DataFrame(summary).sort_values("best_val_huber")
resource_df = pd.DataFrame(per_resource)

summary_df.to_csv(OUT / "model_comparison_summary.csv", index=False)
resource_df.to_csv(OUT / "model_comparison_per_resource.csv", index=False)

print("\n" + "=" * 70)
print("FINAL MODEL COMPARISON — LOWER ERROR IS BETTER")
print("=" * 70)
print(summary_df.to_string(index=False))
print("\nPer-resource results saved to:", OUT / "model_comparison_per_resource.csv")
print("Checkpoints saved to:", CKPTS)
print("Results saved to:", OUT)


Device: cuda
GPU: NVIDIA A100-SXM4-40GB
train: 300,000 windows
val: 50,000 windows
test: 100,000 windows

TRAINING: DLinear


/tmp/ipykernel_6360/1000105098.py:230: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)


DLinear | Epoch 1/8 | train=0.046171 | val=0.036692
DLinear | Epoch 2/8 | train=0.040652 | val=0.036290
DLinear | Epoch 3/8 | train=0.040508 | val=0.036611
DLinear | Epoch 4/8 | train=0.040489 | val=0.035709
DLinear | Epoch 5/8 | train=0.040501 | val=0.035539
DLinear | Epoch 6/8 | train=0.040463 | val=0.035569
DLinear | Epoch 7/8 | train=0.040498 | val=0.035470
DLinear | Epoch 8/8 | train=0.040429 | val=0.035765
Test MAE (scaled): {'CPU usage [%]': np.float64(0.08116), 'Memory usage [KB]': np.float64(0.14135), 'Disk read throughput [KB/s]': np.float64(0.03096), 'Disk write throughput [KB/s]': np.float64(0.04229)}
Test RMSE (scaled): {'CPU usage [%]': np.float64(0.38216), 'Memory usage [KB]': np.float64(0.70241), 'Disk read throughput [KB/s]': np.float64(0.52795), 'Disk write throughput [KB/s]': np.float64(0.72633)}
Test R²: {'CPU usage [%]': np.float64(0.68166), 'Memory usage [KB]': np.float64(0.54159), 'Disk read throughput [KB/s]': np.float64(0.71016), 'Disk write throughput [KB/s]':

In [ ]:

# ============================================================
# NEXT EXPERIMENT: ORIGINAL-UNIT EVALUATION
# Reuses trained models, test_loader and train-only scaler.
# No preprocessing or training is repeated.
# ============================================================

import numpy as np
import pandas as pd
import torch
from pathlib import Path

# Verify required objects from the completed training cell
required = ["models", "test_loader", "scaler", "FEATURES", "DEVICE"]
missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        f"Missing notebook variables: {missing}. "
        "Run this in the same runtime after the completed training cell."
    )

PROJECT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
OUT = PROJECT / "results/forecast_comparison"
OUT.mkdir(parents=True, exist_ok=True)

def evaluate_original_units(model, loader, scaler):
    model.eval()
    n_features = len(FEATURES)

    abs_sum = np.zeros(n_features, dtype=np.float64)
    sq_sum = np.zeros(n_features, dtype=np.float64)
    y_sum = np.zeros(n_features, dtype=np.float64)
    y_sq_sum = np.zeros(n_features, dtype=np.float64)
    count = 0

    with torch.no_grad():
        for x, y_scaled in loader:
            x = x.to(DEVICE)
            pred_scaled = model(x).cpu().numpy()
            actual_scaled = y_scaled.numpy()

            # Invert StandardScaler to recover original resource units
            pred = scaler.inverse_transform(
                pred_scaled.reshape(-1, n_features)
            ).reshape(pred_scaled.shape)

            actual = scaler.inverse_transform(
                actual_scaled.reshape(-1, n_features)
            ).reshape(actual_scaled.shape)

            error = pred - actual

            abs_sum += np.abs(error).sum(axis=(0, 1))
            sq_sum += np.square(error).sum(axis=(0, 1))
            y_sum += actual.sum(axis=(0, 1))
            y_sq_sum += np.square(actual).sum(axis=(0, 1))
            count += actual.shape[0] * actual.shape[1]

    mae = abs_sum / count
    rmse = np.sqrt(sq_sum / count)
    variance = y_sq_sum / count - np.square(y_sum / count)
    r2 = 1 - (sq_sum / count) / np.maximum(variance, 1e-12)

    return mae, rmse, r2


rows = []

for name, model in models.items():
    print(f"\nEvaluating {name} in original units...")
    mae, rmse, r2 = evaluate_original_units(
        model, test_loader, scaler
    )

    for i, feature in enumerate(FEATURES):
        rows.append({
            "model": name,
            "resource": feature,
            "MAE_original_units": float(mae[i]),
            "RMSE_original_units": float(rmse[i]),
            "R2": float(r2[i]),
        })

        print(
            f"{feature}: MAE={mae[i]:.4f}, "
            f"RMSE={rmse[i]:.4f}, R2={r2[i]:.4f}"
        )

results = pd.DataFrame(rows)
results_path = OUT / "original_unit_test_metrics.csv"
results.to_csv(results_path, index=False)

print("\n" + "=" * 75)
print("ORIGINAL-UNIT TEST RESULTS")
print("=" * 75)
print(results.to_string(index=False))
print("\nSaved:", results_path)

print(
    "\nNote: these results use the current capped test sample "
    "of 100,000 windows."
)



Evaluating DLinear in original units...
CPU usage [%]: MAE=1.9105, RMSE=8.9964, R2=0.6817
Memory usage [KB]: MAE=258535.8812, RMSE=1284761.7034, R2=0.5416
Disk read throughput [KB/s]: MAE=161.1686, RMSE=2748.4681, R2=0.7102
Disk write throughput [KB/s]: MAE=46.8658, RMSE=804.9669, R2=0.4282

Evaluating PatchTST in original units...
CPU usage [%]: MAE=1.7746, RMSE=8.6970, R2=0.7025
Memory usage [KB]: MAE=232543.0362, RMSE=1252802.0472, R2=0.5641
Disk read throughput [KB/s]: MAE=163.4289, RMSE=2707.0365, R2=0.7188
Disk write throughput [KB/s]: MAE=45.8963, RMSE=792.7442, R2=0.4455

Evaluating PatchTST_RevIN in original units...
CPU usage [%]: MAE=1.5525, RMSE=8.6923, R2=0.7028
Memory usage [KB]: MAE=225108.4388, RMSE=1257360.8159, R2=0.5609
Disk read throughput [KB/s]: MAE=119.3263, RMSE=2727.8977, R2=0.7145
Disk write throughput [KB/s]: MAE=37.9639, RMSE=824.2667, R2=0.4005

ORIGINAL-UNIT TEST RESULTS
         model                     resource  MAE_original_units  RMSE_original_units 

In [ ]:

# PATCHTST vs PATCHTST+RevIN: HORIZON-WISE EVALUATION
# Reuses existing test_loader and trained models. No training.

import os
import numpy as np
import pandas as pd
import torch

assert "test_loader" in globals(), (
    "Existing test_loader not found. Run the earlier successful 100k test evaluation cell."
)

device = next(iter(trained_models.values())).parameters().__next__().device
features = ["CPU", "Memory", "DiskRead", "DiskWrite"]
steps = [1, 2, 3, 6, 9, 12]  # 5, 10, 15, 30, 45, 60 minutes
rows = []

# Focus research reporting on our main models; DLinear stays only as a baseline.
focus_models = {
    k: v for k, v in trained_models.items()
    if k in ["PatchTST", "PatchTST_RevIN"]
}

for name, model in focus_models.items():
    model.eval()
    sums = {
        h: {"abs": np.zeros(4), "sq": np.zeros(4),
            "n": 0, "sy": np.zeros(4), "sy2": np.zeros(4)}
        for h in steps
    }

    with torch.inference_mode():
        for batch in test_loader:
            xb, yb = batch[0].to(device), batch[1].to(device)
            pred = model(xb)
            if isinstance(pred, (tuple, list)):
                pred = pred[0]

            p = pred.detach().float().cpu().numpy()
            y = yb.detach().float().cpu().numpy()

            p = scaler.inverse_transform(p.reshape(-1, p.shape[-1])).reshape(p.shape)
            y = scaler.inverse_transform(y.reshape(-1, y.shape[-1])).reshape(y.shape)

            for h in steps:
                if h > p.shape[1]:
                    continue
                err = p[:, h-1, :] - y[:, h-1, :]
                actual = y[:, h-1, :]
                s = sums[h]
                s["abs"] += np.abs(err).sum(axis=0)
                s["sq"] += (err**2).sum(axis=0)
                s["sy"] += actual.sum(axis=0)
                s["sy2"] += (actual**2).sum(axis=0)
                s["n"] += len(actual)

    for h, s in sums.items():
        n = max(s["n"], 1)
        mae = s["abs"] / n
        rmse = np.sqrt(s["sq"] / n)
        ss_tot = s["sy2"] - s["sy"]**2 / n
        r2 = np.where(ss_tot > 1e-12, 1 - s["sq"] / ss_tot, np.nan)

        for i, resource in enumerate(features):
            rows.append({
                "Model": name, "Horizon_min": h * 5,
                "Resource": resource, "MAE": mae[i],
                "RMSE": rmse[i], "R2": r2[i]
            })

result = pd.DataFrame(rows)
out_dir = "/content/drive/MyDrive/Bitbrains_Fresh_Research/results/forecast_comparison"
os.makedirs(out_dir, exist_ok=True)
result.to_csv(os.path.join(out_dir, "patchtst_horizon_metrics.csv"), index=False)

display(result.round(4))
print("Saved: patchtst_horizon_metrics.csv")


,Model,Horizon_min,Resource,MAE,RMSE,R2
0,PatchTST,5,CPU,0.7493,4.039800e+00,0.9363
1,PatchTST,5,Memory,138405.7048,7.417871e+05,0.8419
2,PatchTST,5,DiskRead,95.0385,2.840917e+03,0.7634
3,PatchTST,5,DiskWrite,25.6838,4.079991e+02,0.8591
4,PatchTST,10,CPU,1.0999,5.897700e+00,0.8645
5,PatchTST,10,Memory,181238.2079,1.020691e+06,0.7047
6,PatchTST,10,DiskRead,121.4781,2.593516e+03,0.7553
7,PatchTST,10,DiskWrite,32.4382,5.572791e+02,0.7438
8,PatchTST,15,CPU,1.3155,6.943200e+00,0.8118
9,PatchTST,15,Memory,199688.2646,1.112998e+06,0.6527


Saved: patchtst_horizon_metrics.csv


In [ ]:

# PEAK UNDERPREDICTION ANALYSIS — PatchTST vs PatchTST+RevIN
# No retraining. Reuses the existing test_loader and trained models.

import os
import numpy as np
import pandas as pd
import torch

device = next(iter(trained_models["PatchTST"].parameters())).device
resources = ["CPU", "Memory", "DiskRead", "DiskWrite"]
main_models = ["PatchTST", "PatchTST_RevIN"]
horizons = [1, 2, 3, 6, 9, 12]  # 5, 10, 15, 30, 45, 60 min

# Store predictions for the diagnostic evaluation.
pred_store = {m: {h: [] for h in horizons} for m in main_models}
actual_store = {h: [] for h in horizons}

for model_name in main_models:
    model = trained_models[model_name].to(device).eval()

    with torch.inference_mode():
        for batch in test_loader:
            xb, yb = batch[0].to(device), batch[1].to(device)
            pred = model(xb)
            if isinstance(pred, (tuple, list)):
                pred = pred[0]

            p = pred.detach().float().cpu().numpy()
            y = yb.detach().float().cpu().numpy()

            p = scaler.inverse_transform(p.reshape(-1, p.shape[-1])).reshape(p.shape)
            y = scaler.inverse_transform(y.reshape(-1, y.shape[-1])).reshape(y.shape)

            for h in horizons:
                if h <= p.shape[1]:
                    pred_store[model_name][h].append(p[:, h - 1, :])

                    # Save actuals only once; they are the same for both models.
                    if model_name == main_models[0]:
                        actual_store[h].append(y[:, h - 1, :])

rows = []

for h in horizons:
    actual = np.concatenate(actual_store[h], axis=0)

    for model_name in main_models:
        pred = np.concatenate(pred_store[model_name][h], axis=0)

        for j, resource in enumerate(resources):
            y = actual[:, j]
            p = pred[:, j]
            err = p - y

            # Diagnostic peak threshold: top 5% of actual values in this
            # test sample. This is NOT a deployment threshold.
            peak_threshold = np.quantile(y, 0.95)
            peak_mask = y >= peak_threshold

            under = err < 0
            peak_under = peak_mask & under

            rows.append({
                "Model": model_name,
                "Horizon_min": h * 5,
                "Resource": resource,
                "Underprediction_rate_pct": 100 * under.mean(),
                "Mean_signed_error": err.mean(),
                "Underprediction_MAE": (
                    np.abs(err[under]).mean() if under.any() else 0.0
                ),
                "Actual_peak_threshold_p95": peak_threshold,
                "Peak_underprediction_rate_pct": (
                    100 * peak_under.sum() / max(peak_mask.sum(), 1)
                ),
                "Peak_MAE": (
                    np.abs(err[peak_mask]).mean() if peak_mask.any() else np.nan
                ),
                "Peak_underprediction_MAE": (
                    np.abs(err[peak_under]).mean()
                    if peak_under.any() else 0.0
                ),
                "Peak_observations": int(peak_mask.sum())
            })

peak_df = pd.DataFrame(rows)
out_dir = "/content/drive/MyDrive/Bitbrains_Fresh_Research/results/forecast_comparison"
os.makedirs(out_dir, exist_ok=True)
save_path = os.path.join(out_dir, "peak_underprediction_analysis.csv")
peak_df.to_csv(save_path, index=False)

print("CPU + Memory: peak-risk comparison")
display(
    peak_df[peak_df["Resource"].isin(["CPU", "Memory"])][[
        "Model", "Horizon_min", "Resource",
        "Underprediction_rate_pct", "Mean_signed_error",
        "Peak_underprediction_rate_pct", "Peak_MAE",
        "Peak_underprediction_MAE"
    ]].round(4)
)

print("Saved:", save_path)


CPU + Memory: peak-risk comparison


,Model,Horizon_min,Resource,Underprediction_rate_pct,Mean_signed_error,Peak_underprediction_rate_pct,Peak_MAE,Peak_underprediction_MAE
0,PatchTST,5,CPU,30.416,-0.018800,60.4437,7.741200e+00,9.137600e+00
1,PatchTST,5,Memory,37.824,-16034.406250,60.3079,1.593598e+06,1.812468e+06
4,PatchTST_RevIN,5,CPU,30.015,-0.013700,49.8501,7.938000e+00,1.121810e+01
5,PatchTST_RevIN,5,Memory,38.645,-12605.829102,56.6687,1.565475e+06,1.899289e+06
8,PatchTST,10,CPU,21.033,-0.137700,80.0360,1.224420e+01,1.273030e+01
9,PatchTST,10,Memory,31.427,-36040.519531,70.1120,2.182358e+06,2.415862e+06
12,PatchTST_RevIN,10,CPU,37.723,-0.135800,52.3981,1.170840e+01,1.668900e+01
13,PatchTST_RevIN,10,Memory,42.575,-36854.933594,61.9752,2.185190e+06,2.606404e+06
16,PatchTST,15,CPU,27.671,-0.291400,84.9800,1.530520e+01,1.573320e+01
17,PatchTST,15,Memory,36.577,-56512.460938,76.3084,2.437166e+06,2.625435e+06


Saved: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/forecast_comparison/peak_underprediction_analysis.csv


In [ ]:

# VALIDATION-CALIBRATED UPPER FORECASTS
# Goal: reduce underprediction without retraining PatchTST or RevIN.
# Calibration: 95th percentile of (actual - prediction) on VALIDATION only.
# Test data is used only for final evaluation.

import os
import numpy as np
import pandas as pd
import torch

assert "val_loader" in globals(), "Need the existing validation DataLoader (val_loader)."
assert "test_loader" in globals(), "Need the existing test DataLoader (test_loader)."
assert "scaler" in globals(), "Train-only fitted scaler is required."

device = next(iter(trained_models["PatchTST"].parameters())).device
model_names = ["PatchTST", "PatchTST_RevIN"]
features = ["CPU", "Memory", "DiskRead", "DiskWrite"]
horizons = [1, 2, 3, 6, 9, 12]
CALIBRATION_QUANTILE = 0.95

def collect_predictions(model, loader):
    model.eval()
    predictions, actuals = [], []

    with torch.inference_mode():
        for batch in loader:
            xb, yb = batch[0].to(device), batch[1].to(device)
            p = model(xb)
            if isinstance(p, (tuple, list)):
                p = p[0]

            p = p.detach().float().cpu().numpy()
            y = yb.detach().float().cpu().numpy()

            p = scaler.inverse_transform(p.reshape(-1, p.shape[-1])).reshape(p.shape)
            y = scaler.inverse_transform(y.reshape(-1, y.shape[-1])).reshape(y.shape)

            predictions.append(p)
            actuals.append(y)

    return np.concatenate(predictions), np.concatenate(actuals)

calibration_rows = []
calibration_results = []

for model_name in model_names:
    model = trained_models[model_name].to(device)

    print(f"Calibrating {model_name} on validation data...")
    val_pred, val_true = collect_predictions(model, val_loader)

    print(f"Evaluating {model_name} on test data...")
    test_pred, test_true = collect_predictions(model, test_loader)

    for h in horizons:
        idx = h - 1
        if idx >= val_pred.shape[1] or idx >= test_pred.shape[1]:
            continue

        # Positive residual means the model underpredicted actual usage.
        val_residual = val_true[:, idx, :] - val_pred[:, idx, :]

        # Resource-specific upper correction learned ONLY from validation.
        margins = np.quantile(
            val_residual, CALIBRATION_QUANTILE, axis=0
        )

        y = test_true[:, idx, :]
        raw = test_pred[:, idx, :]
        calibrated = raw + margins

        for j, resource in enumerate(features):
            peak_threshold = np.quantile(y[:, j], 0.95)
            peak_mask = y[:, j] >= peak_threshold

            for method, prediction in [
                ("Raw", raw[:, j]),
                ("Calibrated_95", calibrated[:, j])
            ]:
                err = prediction - y[:, j]
                under = err < 0
                peak_under = peak_mask & under

                calibration_results.append({
                    "Model": model_name,
                    "Horizon_min": h * 5,
                    "Resource": resource,
                    "Method": method,
                    "Underprediction_pct": 100 * under.mean(),
                    "Peak_underprediction_pct": (
                        100 * peak_under.sum() / max(peak_mask.sum(), 1)
                    ),
                    "MAE": np.abs(err).mean(),
                    "RMSE": np.sqrt(np.mean(err ** 2)),
                    "Mean_overprediction": np.maximum(err, 0).mean(),
                    "Calibration_margin": (
                        margins[j] if method == "Calibrated_95" else 0.0
                    ),
                    "Test_peak_threshold_p95": peak_threshold,
                    "Test_windows": len(y)
                })

            calibration_rows.append({
                "Model": model_name,
                "Horizon_min": h * 5,
                "Resource": resource,
                "Validation_residual_q95": margins[j]
            })

    del val_pred, val_true, test_pred, test_true

out_dir = "/content/drive/MyDrive/Bitbrains_Fresh_Research/results/forecast_comparison"
os.makedirs(out_dir, exist_ok=True)

result_df = pd.DataFrame(calibration_results)
margin_df = pd.DataFrame(calibration_rows)

result_df.to_csv(
    os.path.join(out_dir, "calibrated_forecast_test_metrics.csv"), index=False
)
margin_df.to_csv(
    os.path.join(out_dir, "validation_calibration_margins.csv"), index=False
)

print("\n=== UNDERPREDICTION COMPARISON ===")
display(
    result_df.pivot_table(
        index=["Model", "Horizon_min", "Resource"],
        columns="Method",
        values=["Underprediction_pct", "Peak_underprediction_pct", "MAE"],
    ).round(3)
)

print("\n=== VALIDATION-LEARNED SAFETY MARGINS ===")
display(margin_df.round(3))

print("\nSaved calibrated metrics and validation-only margins in:", out_dir)


Calibrating PatchTST on validation data...
Evaluating PatchTST on test data...
Calibrating PatchTST_RevIN on validation data...
Evaluating PatchTST_RevIN on test data...

=== UNDERPREDICTION COMPARISON ===


MAE                 \
Method                                Calibrated_95            Raw   
Model          Horizon_min Resource                                  
PatchTST       5           CPU             1.537000       0.749000   
                           DiskRead       94.274002      95.038002   
                           DiskWrite      31.476000      25.684000   
                           Memory     325638.750000  138405.718750   
               10          CPU             4.566000       1.100000   
                           DiskRead      107.606003     121.477997   
                           DiskWrite      36.342999      32.438000   
                           Memory     454654.031250  181238.203125   
               15          CPU             7.048000       1.315000   
                           DiskRead      117.612000     193.804001   
                           DiskWrite      41.619999      47.000999   
                           Memory     529633.812500  199688.234375   
               30          CPU             4.597000       1.770000   
                           DiskRead      122.042999     197.550003   
                           DiskWrite      46.058998      54.478001   
                           Memory     523795.593750  236971.453125   
               45          CPU             4.528000       2.302000   
                           DiskRead      129.544998     203.764999   
                           DiskWrite      48.800999      58.025002   
                           Memory     537603.187500  271291.812500   
               60          CPU             6.593000       2.512000   
                           DiskRead      132.248001     157.481995   
                           DiskWrite      51.972000      48.436001   
                           Memory     618946.875000  280935.906250   
PatchTST_RevIN 5           CPU             1.223000       0.744000   
                           DiskRead       97.692001      97.056000   
                           DiskWrite      30.115999      25.913000   
                           Memory     325979.437500  139490.437500   
               10          CPU             1.680000       0.998000   
                           DiskRead      103.688004     101.113998   
                           DiskWrite      35.325001      28.806000   
                           Memory     421444.937500  179790.281250   
               15          CPU             2.239000       1.182000   
                           DiskRead      114.709999     111.960999   
                           DiskWrite      38.365002      30.882999   
                           Memory     454562.406250  198477.234375   
               30          CPU             3.102000       1.566000   
                           DiskRead      119.390999     115.862000   
                           DiskWrite      47.144001      39.275002   
                           Memory     514009.406250  228683.265625   
               45          CPU             3.457000       1.832000   
                           DiskRead      130.764999     127.276001   
                           DiskWrite      51.945999      43.813999   
                           Memory     544068.187500  250747.781250   
               60          CPU             4.320000       2.105000   
                           DiskRead      132.639008     129.647995   
                           DiskWrite      53.151001      44.618000   
                           Memory     574656.812500  266780.031250   

                                     Peak_underprediction_pct          \
Method                                          Calibrated_95     Raw   
Model          Horizon_min Resource                                     
PatchTST       5           CPU                         42.554  60.444   
                           DiskRead                    52.508  56.366   
                           DiskWrite                   44.233  53.288   
                           Memory      


=== VALIDATION-LEARNED SAFETY MARGINS ===


,Model,Horizon_min,Resource,Validation_residual_q95
0,PatchTST,5,CPU,0.946000
1,PatchTST,5,Memory,236017.546875
2,PatchTST,5,DiskRead,5.438000
3,PatchTST,5,DiskWrite,7.846000
4,PatchTST,10,CPU,3.859000
5,PatchTST,10,Memory,334184.250000
6,PatchTST,10,DiskRead,19.559000
7,PatchTST,10,DiskWrite,11.357000
8,PatchTST,15,CPU,6.363000
9,PatchTST,15,Memory,406148.468750



Saved calibrated metrics and validation-only margins in: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/forecast_comparison


In [ ]:

# LOAD-DEPENDENT RESIDUAL CALIBRATION
# Learn corrections from validation predictions; evaluate only on test data.
# No model retraining.

import os
import numpy as np
import pandas as pd
import torch

device = next(iter(trained_models["PatchTST"].parameters())).device
model_names = ["PatchTST", "PatchTST_RevIN"]
resources = ["CPU", "Memory", "DiskRead", "DiskWrite"]
horizons = [1, 2, 3, 6, 9, 12]
Q = 0.95
N_BINS = 10
MIN_BIN_SAMPLES = 200

def get_original_unit_predictions(model, loader):
    model.eval()
    ps, ys = [], []

    with torch.inference_mode():
        for batch in loader:
            xb, yb = batch[0].to(device), batch[1].to(device)
            p = model(xb)
            if isinstance(p, (tuple, list)):
                p = p[0]

            p = p.detach().float().cpu().numpy()
            y = yb.detach().float().cpu().numpy()

            p = scaler.inverse_transform(p.reshape(-1, p.shape[-1])).reshape(p.shape)
            y = scaler.inverse_transform(y.reshape(-1, y.shape[-1])).reshape(y.shape)
            ps.append(p)
            ys.append(y)

    return np.concatenate(ps), np.concatenate(ys)

def learn_bin_corrections(val_pred, val_true):
    # Corrections vary by horizon and resource, according to predicted load.
    corrections = {}

    for h in horizons:
        idx = h - 1
        if idx >= val_pred.shape[1]:
            continue

        for j, resource in enumerate(resources):
            p = val_pred[:, idx, j]
            residual = val_true[:, idx, j] - p

            # Equal-frequency bins based on VALIDATION predictions only.
            edges = np.unique(np.quantile(p, np.linspace(0, 1, N_BINS + 1)))
            if len(edges) < 3:
                edges = np.array([-np.inf, np.inf])
            else:
                edges[0], edges[-1] = -np.inf, np.inf

            bin_margins = []
            global_margin = np.quantile(residual, Q)

            for b in range(len(edges) - 1):
                mask = (p >= edges[b]) & (p < edges[b + 1])
                if mask.sum() >= MIN_BIN_SAMPLES:
                    margin = np.quantile(residual[mask], Q)
                else:
                    # Sparse bins fall back to the validation-wide correction.
                    margin = global_margin
                bin_margins.append(float(margin))

            corrections[(h, j)] = {
                "edges": edges,
                "margins": np.array(bin_margins),
                "global_margin": float(global_margin)
            }

    return corrections

def apply_bin_corrections(pred, corrections):
    calibrated = pred.copy()

    for h in horizons:
        idx = h - 1
        if idx >= pred.shape[1]:
            continue

        for j in range(pred.shape[-1]):
            info = corrections[(h, j)]
            edges, margins = info["edges"], info["margins"]
            bin_ids = np.searchsorted(edges[1:-1], pred[:, idx, j], side="right")
            bin_ids = np.clip(bin_ids, 0, len(margins) - 1)
            calibrated[:, idx, j] += margins[bin_ids]

    return calibrated

rows = []
margin_rows = []

for model_name in model_names:
    model = trained_models[model_name].to(device)

    print("Validation:", model_name)
    val_pred, val_true = get_original_unit_predictions(model, val_loader)
    corrections = learn_bin_corrections(val_pred, val_true)

    print("Test:", model_name)
    test_pred, test_true = get_original_unit_predictions(model, test_loader)
    cal_pred = apply_bin_corrections(test_pred, corrections)

    for h in horizons:
        idx = h - 1
        if idx >= test_pred.shape[1]:
            continue

        for j, resource in enumerate(resources):
            y = test_true[:, idx, j]
            raw = test_pred[:, idx, j]
            calibrated = cal_pred[:, idx, j]
            peak_cutoff = np.quantile(y, 0.95)
            peak_mask = y >= peak_cutoff

            for method, p in [
                ("Raw", raw),
                ("LoadCalibrated_95", calibrated)
            ]:
                err = p - y
                under = err < 0
                peak_under = peak_mask & under

                rows.append({
                    "Model": model_name,
                    "Horizon_min": h * 5,
                    "Resource": resource,
                    "Method": method,
                    "MAE": np.abs(err).mean(),
                    "RMSE": np.sqrt(np.mean(err ** 2)),
                    "Underprediction_pct": 100 * under.mean(),
                    "Peak_underprediction_pct": (
                        100 * peak_under.sum() / max(peak_mask.sum(), 1)
                    ),
                    "Peak_MAE": np.abs(err[peak_mask]).mean(),
                    "Mean_overprediction": np.maximum(err, 0).mean()
                })

            info = corrections[(h, j)]
            for b, margin in enumerate(info["margins"]):
                margin_rows.append({
                    "Model": model_name,
                    "Horizon_min": h * 5,
                    "Resource": resource,
                    "Predicted_load_bin": b,
                    "Validation_residual_margin": margin,
                    "Validation_global_margin": info["global_margin"]
                })

out_dir = "/content/drive/MyDrive/Bitbrains_Fresh_Research/results/forecast_comparison"
os.makedirs(out_dir, exist_ok=True)

metrics_df = pd.DataFrame(rows)
margins_df = pd.DataFrame(margin_rows)

metrics_df.to_csv(
    os.path.join(out_dir, "load_dependent_calibration_metrics.csv"), index=False
)
margins_df.to_csv(
    os.path.join(out_dir, "load_dependent_calibration_margins.csv"), index=False
)

print("\n=== CPU AND MEMORY: RAW VS LOAD-DEPENDENT CALIBRATION ===")
display(
    metrics_df[metrics_df["Resource"].isin(["CPU", "Memory"])][[
        "Model", "Horizon_min", "Resource", "Method",
        "MAE", "Peak_underprediction_pct", "Peak_MAE",
        "Mean_overprediction"
    ]].round(3)
)

print("\nSaved results to:", out_dir)


Validation: PatchTST
Test: PatchTST
Validation: PatchTST_RevIN
Test: PatchTST_RevIN

=== CPU AND MEMORY: RAW VS LOAD-DEPENDENT CALIBRATION ===


,Model,Horizon_min,Resource,Method,MAE,Peak_underprediction_pct,Peak_MAE,Mean_overprediction
0,PatchTST,5,CPU,Raw,0.749000,60.444,7.741000e+00,0.365000
1,PatchTST,5,CPU,LoadCalibrated_95,1.369000,16.530,1.087000e+01,1.136000
2,PatchTST,5,Memory,Raw,138405.718750,60.308,1.593598e+06,61185.648438
3,PatchTST,5,Memory,LoadCalibrated_95,317087.468750,15.417,2.710949e+06,281758.468750
8,PatchTST,10,CPU,Raw,1.100000,80.036,1.224400e+01,0.481000
9,PatchTST,10,CPU,LoadCalibrated_95,1.871000,19.704,1.539600e+01,1.501000
10,PatchTST,10,Memory,Raw,181238.203125,70.112,2.182358e+06,72598.843750
11,PatchTST,10,Memory,LoadCalibrated_95,418333.812500,19.712,3.639138e+06,365573.125000
16,PatchTST,15,CPU,Raw,1.315000,84.980,1.530500e+01,0.512000
17,PatchTST,15,CPU,LoadCalibrated_95,2.184000,22.140,1.825200e+01,1.729000



Saved results to: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/forecast_comparison


In [ ]:

# DECISION-LEVEL PROXY: HIGH-RESOURCE EVENT DETECTION
# Validation determines thresholds; test measures detection quality.
# This is a high-load detection proxy, not yet a physical VM consolidation simulator.

import os
import numpy as np
import pandas as pd
import torch

device = next(iter(trained_models["PatchTST"].parameters())).device
models_to_eval = ["PatchTST", "PatchTST_RevIN"]
resources = ["CPU", "Memory", "DiskRead", "DiskWrite"]
horizons = [1, 2, 3, 6, 9, 12]
HIGH_LOAD_QUANTILE = 0.95

# Reuse the prediction collector from the previous cell.
rows = []

for model_name in models_to_eval:
    model = trained_models[model_name].to(device)

    print("Validation thresholds:", model_name)
    val_pred, val_true = get_original_unit_predictions(model, val_loader)
    test_pred, test_true = get_original_unit_predictions(model, test_loader)

    for h in horizons:
        idx = h - 1
        if idx >= test_pred.shape[1] or idx >= val_pred.shape[1]:
            continue

        for j, resource in enumerate(resources):
            # Define high load from validation ACTUALS only.
            threshold = np.quantile(val_true[:, idx, j], HIGH_LOAD_QUANTILE)

            actual_high = test_true[:, idx, j] >= threshold
            raw_high = test_pred[:, idx, j] >= threshold

            # Evaluate the existing load-dependent correction using
            # margins fitted on validation data only.
            # Relearn its corrections here, never using test labels.
            val_residual = val_true[:, idx, j] - val_pred[:, idx, j]
            edges = np.unique(
                np.quantile(val_pred[:, idx, j], np.linspace(0, 1, 11))
            )

            if len(edges) >= 3:
                edges[0], edges[-1] = -np.inf, np.inf
                bin_ids_val = np.searchsorted(
                    edges[1:-1], val_pred[:, idx, j], side="right"
                )
                bin_ids_test = np.searchsorted(
                    edges[1:-1], test_pred[:, idx, j], side="right"
                )
                bin_ids_val = np.clip(bin_ids_val, 0, len(edges) - 2)
                bin_ids_test = np.clip(bin_ids_test, 0, len(edges) - 2)

                global_margin = np.quantile(val_residual, 0.95)
                margins = []
                for b in range(len(edges) - 1):
                    mask = bin_ids_val == b
                    margins.append(
                        np.quantile(val_residual[mask], 0.95)
                        if mask.sum() >= 200 else global_margin
                    )
                margins = np.asarray(margins)
                calibrated_pred = (
                    test_pred[:, idx, j] + margins[bin_ids_test]
                )
            else:
                calibrated_pred = (
                    test_pred[:, idx, j] + np.quantile(val_residual, 0.95)
                )

            for method, predicted_high in [
                ("Raw", raw_high),
                ("LoadCalibrated_95", calibrated_pred >= threshold)
            ]:
                tp = np.sum(predicted_high & actual_high)
                fp = np.sum(predicted_high & ~actual_high)
                fn = np.sum(~predicted_high & actual_high)
                tn = np.sum(~predicted_high & ~actual_high)

                precision = tp / max(tp + fp, 1)
                recall = tp / max(tp + fn, 1)
                f1 = (
                    2 * precision * recall / (precision + recall)
                    if precision + recall else 0.0
                )

                rows.append({
                    "Model": model_name,
                    "Horizon_min": h * 5,
                    "Resource": resource,
                    "Method": method,
                    "Validation_high_load_threshold": threshold,
                    "Precision_pct": 100 * precision,
                    "Recall_pct": 100 * recall,
                    "F1_pct": 100 * f1,
                    "Missed_high_load_pct": 100 * fn / max(tp + fn, 1),
                    "False_alarm_pct": 100 * fp / max(fp + tn, 1),
                    "Predicted_high_load_pct": 100 * predicted_high.mean()
                })

    del val_pred, val_true, test_pred, test_true

decision_df = pd.DataFrame(rows)
out_dir = "/content/drive/MyDrive/Bitbrains_Fresh_Research/results/forecast_comparison"
os.makedirs(out_dir, exist_ok=True)
path = os.path.join(out_dir, "high_load_detection_metrics.csv")
decision_df.to_csv(path, index=False)

print("\n=== CPU AND MEMORY HIGH-LOAD DETECTION ===")
display(
    decision_df[decision_df["Resource"].isin(["CPU", "Memory"])][[
        "Model", "Horizon_min", "Resource", "Method",
        "Precision_pct", "Recall_pct", "F1_pct",
        "Missed_high_load_pct", "False_alarm_pct",
        "Predicted_high_load_pct"
    ]].round(2)
)
print("\nSaved:", path)


Validation thresholds: PatchTST
Validation thresholds: PatchTST_RevIN

=== CPU AND MEMORY HIGH-LOAD DETECTION ===


,Model,Horizon_min,Resource,Method,Precision_pct,Recall_pct,F1_pct,Missed_high_load_pct,False_alarm_pct,Predicted_high_load_pct
0,PatchTST,5,CPU,Raw,92.97,88.67,90.77,11.33,0.15,2.02
1,PatchTST,5,CPU,LoadCalibrated_95,89.30,91.83,90.55,8.17,0.24,2.18
2,PatchTST,5,Memory,Raw,90.81,89.06,89.92,10.94,0.25,2.69
3,PatchTST,5,Memory,LoadCalibrated_95,57.01,95.26,71.33,4.74,2.02,4.58
8,PatchTST,10,CPU,Raw,89.54,74.25,81.18,25.75,0.19,1.75
9,PatchTST,10,CPU,LoadCalibrated_95,84.33,85.96,85.14,14.04,0.34,2.15
10,PatchTST,10,Memory,Raw,87.17,80.40,83.65,19.60,0.34,2.56
11,PatchTST,10,Memory,LoadCalibrated_95,36.65,92.35,52.47,7.65,4.55,6.98
16,PatchTST,15,CPU,Raw,86.88,66.51,75.34,33.49,0.22,1.62
17,PatchTST,15,CPU,LoadCalibrated_95,80.32,81.60,80.95,18.40,0.43,2.15



Saved: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/forecast_comparison/high_load_detection_metrics.csv


In [ ]:

# CONSOLIDATION SIMULATOR — DATA READINESS CHECK
# Read-only inspection. Does not modify miniPatch or retrain models.

from pathlib import Path
import pandas as pd
import os

ROOT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
print("Project exists:", ROOT.exists())
print("Project:", ROOT)

for folder in [
    ROOT / "data" / "splits" / "train",
    ROOT / "data" / "splits" / "val",
    ROOT / "data" / "splits" / "test",
    ROOT / "data" / "processed" / "bitbrains_5min",
]:
    print(f"\n{folder}")
    if not folder.exists():
        print("  NOT FOUND")
        continue

    files = list(folder.glob("*.csv"))
    print("  CSV files:", len(files))

    if files:
        sample = files[0]
        print("  Sample file:", sample.name)
        try:
            df = pd.read_csv(sample, nrows=5)
            print("  Columns:", df.columns.tolist())
            print(df.head(2).to_string(index=False))
        except Exception as e:
            print("  Read error:", repr(e))

# Check the project output files already created.
results = ROOT / "results" / "forecast_comparison"
print("\nExisting experiment outputs:")
if results.exists():
    for f in sorted(results.glob("*.csv")):
        print(" ", f.name)
else:
    print("Results folder not found.")


Project exists: True
Project: /content/drive/MyDrive/Bitbrains_Fresh_Research

/content/drive/MyDrive/Bitbrains_Fresh_Research/data/splits/train
  CSV files: 1217
  Sample file: 222.csv
  Columns: ['Timestamp', 'VM_ID', 'CPU usage [%]', 'Memory usage [KB]', 'Disk read throughput [KB/s]', 'Disk write throughput [KB/s]']
          Timestamp  VM_ID  CPU usage [%]  Memory usage [KB]  Disk read throughput [KB/s]  Disk write throughput [KB/s]
2013-08-16 12:00:00    222            0.0                0.0                          0.0                           0.0
2013-08-16 12:05:00    222            0.0                0.0                          0.0                           0.0

/content/drive/MyDrive/Bitbrains_Fresh_Research/data/splits/val
  CSV files: 1217
  Sample file: 222.csv
  Columns: ['Timestamp', 'VM_ID', 'CPU usage [%]', 'Memory usage [KB]', 'Disk read throughput [KB/s]', 'Disk write throughput [KB/s]']
          Timestamp  VM_ID  CPU usage [%]  Memory usage [KB]  Disk read throug

In [ ]:

# MULTI-RESOURCE VM CONSOLIDATION — ORACLE BASELINE
# Read-only input data. No model training. Does not touch miniPatch.
# IMPORTANT: host capacities below are configurable assumptions, not dataset facts.

from pathlib import Path
import numpy as np
import pandas as pd
import os

ROOT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
TEST_DIR = ROOT / "data" / "splits" / "test"
OUT_DIR = ROOT / "results" / "consolidation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Assumed host configuration — change these to match your intended deployment.
HOST_CPU_CAPACITY = 800.0            # e.g. 8 cores x 100% per core
HOST_MEMORY_CAPACITY_KB = 64 * 1024 * 1024  # 64 GiB in KiB
SAFETY_FACTOR = 0.80                 # use at most 80% of each host's capacity
SAMPLE_EVERY_N_TIMESTAMPS = 12       # sample every 60 minutes from 5-min data
MIN_ACTIVE_VMS = 20                  # ignore timestamps with fewer VMs available

CPU_COL = "CPU usage [%]"
MEM_COL = "Memory usage [KB]"

# 1. Load each VM's held-out test trace.
traces = []
for file in TEST_DIR.glob("*.csv"):
    try:
        df = pd.read_csv(file, usecols=["Timestamp", "VM_ID", CPU_COL, MEM_COL])
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
        df[CPU_COL] = pd.to_numeric(df[CPU_COL], errors="coerce")
        df[MEM_COL] = pd.to_numeric(df[MEM_COL], errors="coerce")
        df = df.dropna(subset=["Timestamp", "VM_ID", CPU_COL, MEM_COL])
        df = df[(df[CPU_COL] >= 0) & (df[MEM_COL] >= 0)]
        traces.append(df)
    except Exception as e:
        print(f"Skipped {file.name}: {e}")

if not traces:
    raise RuntimeError(f"No usable test traces found in {TEST_DIR}")

data = pd.concat(traces, ignore_index=True)
data = data.drop_duplicates(["Timestamp", "VM_ID"], keep="last")

print("VMs loaded:", data["VM_ID"].nunique())
print("Rows loaded:", len(data))
print("Time range:", data["Timestamp"].min(), "to", data["Timestamp"].max())

# 2. Sample aligned snapshots. Each snapshot only includes VMs with an
# actual observation at that exact timestamp; no forward-filling.
all_times = np.sort(data["Timestamp"].unique())
sampled_times = set(all_times[::SAMPLE_EVERY_N_TIMESTAMPS])
data = data[data["Timestamp"].isin(sampled_times)]

cpu_cap = HOST_CPU_CAPACITY * SAFETY_FACTOR
mem_cap = HOST_MEMORY_CAPACITY_KB * SAFETY_FACTOR

# 3. First-Fit Decreasing using dominant normalized resource demand.
# A VM is placed only if both CPU and memory remain within effective capacity.
def pack_ffd(snapshot):
    vms = snapshot[["VM_ID", CPU_COL, MEM_COL]].copy()
    vms = vms.rename(columns={CPU_COL: "cpu", MEM_COL: "mem"})

    # Ignore zero-demand VMs for sorting, but retain them for placement.
    vms["dominant"] = np.maximum(
        vms["cpu"] / cpu_cap,
        vms["mem"] / mem_cap
    )
    vms = vms.sort_values("dominant", ascending=False)

    hosts = []  # each: [cpu_used, memory_used, vm_count]
    unplaceable = 0

    for row in vms.itertuples(index=False):
        vm_id, cpu, mem, _ = row

        # A VM that exceeds either effective host capacity cannot fit alone.
        if cpu > cpu_cap or mem > mem_cap:
            unplaceable += 1
            continue

        placed = False
        for host in hosts:
            if host[0] + cpu <= cpu_cap and host[1] + mem <= mem_cap:
                host[0] += cpu
                host[1] += mem
                host[2] += 1
                placed = True
                break

        if not placed:
            hosts.append([float(cpu), float(mem), 1])

    if not hosts:
        return {
            "hosts_used": 0, "unplaceable_vms": unplaceable,
            "cpu_util_pct": np.nan, "memory_util_pct": np.nan,
            "hosts_over_80pct_cpu": 0, "hosts_over_80pct_memory": 0
        }

    host_array = np.asarray(hosts, dtype=float)
    cpu_util = host_array[:, 0] / cpu_cap * 100
    mem_util = host_array[:, 1] / mem_cap * 100

    return {
        "hosts_used": len(hosts),
        "unplaceable_vms": unplaceable,
        "cpu_util_pct": cpu_util.mean(),
        "memory_util_pct": mem_util.mean(),
        "hosts_over_80pct_cpu": int((cpu_util > 80).sum()),
        "hosts_over_80pct_memory": int((mem_util > 80).sum())
    }

# 4. Simulate snapshots and save results.
snapshot_rows = []
for timestamp, snap in data.groupby("Timestamp", sort=True):
    if snap["VM_ID"].nunique() < MIN_ACTIVE_VMS:
        continue

    result = pack_ffd(snap)
    result["Timestamp"] = timestamp
    result["vms_observed"] = snap["VM_ID"].nunique()
    result["total_cpu_demand_pct"] = snap[CPU_COL].sum()
    result["total_memory_demand_kb"] = snap[MEM_COL].sum()
    snapshot_rows.append(result)

sim_df = pd.DataFrame(snapshot_rows)
if sim_df.empty:
    raise RuntimeError(
        "No snapshots passed MIN_ACTIVE_VMS. Lower MIN_ACTIVE_VMS or "
        "inspect timestamp overlap before proceeding."
    )

sim_path = OUT_DIR / "oracle_ffd_snapshot_results.csv"
sim_df.to_csv(sim_path, index=False)

summary = pd.DataFrame([{
    "snapshots_evaluated": len(sim_df),
    "mean_vms_per_snapshot": sim_df["vms_observed"].mean(),
    "mean_hosts_used": sim_df["hosts_used"].mean(),
    "median_hosts_used": sim_df["hosts_used"].median(),
    "p95_hosts_used": sim_df["hosts_used"].quantile(0.95),
    "mean_cpu_util_pct": sim_df["cpu_util_pct"].mean(),
    "mean_memory_util_pct": sim_df["memory_util_pct"].mean(),
    "snapshots_with_unplaceable_vms_pct":
        100 * (sim_df["unplaceable_vms"] > 0).mean(),
    "mean_unplaceable_vms": sim_df["unplaceable_vms"].mean(),
    "assumed_host_cpu_capacity_pct": HOST_CPU_CAPACITY,
    "assumed_host_memory_capacity_kb": HOST_MEMORY_CAPACITY_KB,
    "safety_factor": SAFETY_FACTOR
}])

summary_path = OUT_DIR / "oracle_ffd_summary.csv"
summary.to_csv(summary_path, index=False)

print("\n=== ORACLE FFD BASELINE SUMMARY ===")
display(summary.round(3))
print("Saved snapshots:", sim_path)
print("Saved summary:", summary_path)


VMs loaded: 1217
Rows loaded: 1541556
Time range: 2013-08-20 01:20:00 to 2013-09-11 13:35:00

=== ORACLE FFD BASELINE SUMMARY ===


,snapshots_evaluated,mean_vms_per_snapshot,mean_hosts_used,median_hosts_used,p95_hosts_used,mean_cpu_util_pct,mean_memory_util_pct,snapshots_with_unplaceable_vms_pct,mean_unplaceable_vms,assumed_host_cpu_capacity_pct,assumed_host_memory_capacity_kb,safety_factor
0,108,1179.843,11.639,10.0,21.65,79.285,89.604,0.926,0.009,800.0,67108864,0.8


Saved snapshots: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/consolidation/oracle_ffd_snapshot_results.csv
Saved summary: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/consolidation/oracle_ffd_summary.csv


In [ ]:

# HOST CAPACITY SENSITIVITY ANALYSIS
# Reuses `data` and `pack_ffd` from the previous cell.
# No model training; no source data modification.

import numpy as np
import pandas as pd
from pathlib import Path

assert "data" in globals() and "pack_ffd" in globals(), (
    "Run the oracle FFD baseline cell first."
)

ROOT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
OUT_DIR = ROOT / "results" / "consolidation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PHYSICAL_CPU_CAPACITY = 800.0
PHYSICAL_MEMORY_CAPACITY_KB = 64 * 1024 * 1024

# Compare the 80% setting used previously against tighter and looser limits.
safety_factors = [0.60, 0.70, 0.80, 0.90]
sensitivity_rows = []

snapshots = [
    (timestamp, snap)
    for timestamp, snap in data.groupby("Timestamp", sort=True)
    if snap["VM_ID"].nunique() >= 20
]

for factor in safety_factors:
    # pack_ffd uses these global effective capacities.
    cpu_cap = PHYSICAL_CPU_CAPACITY * factor
    mem_cap = PHYSICAL_MEMORY_CAPACITY_KB * factor

    host_counts = []
    cpu_utils = []
    mem_utils = []
    unplaceable_counts = []

    for timestamp, snap in snapshots:
        result = pack_ffd(snap)
        host_counts.append(result["hosts_used"])
        cpu_utils.append(result["cpu_util_pct"])
        mem_utils.append(result["memory_util_pct"])
        unplaceable_counts.append(result["unplaceable_vms"])

    sensitivity_rows.append({
        "Safety_factor": factor,
        "Snapshots": len(snapshots),
        "Mean_hosts": np.mean(host_counts),
        "Median_hosts": np.median(host_counts),
        "P95_hosts": np.quantile(host_counts, 0.95),
        "Mean_CPU_util_pct_of_effective_capacity": np.nanmean(cpu_utils),
        "Mean_memory_util_pct_of_effective_capacity": np.nanmean(mem_utils),
        "Snapshots_with_unplaceable_VMs_pct":
            100 * np.mean(np.asarray(unplaceable_counts) > 0),
        "Total_unplaceable_VM_snapshot_events": int(sum(unplaceable_counts))
    })

sensitivity_df = pd.DataFrame(sensitivity_rows)
save_path = OUT_DIR / "capacity_sensitivity.csv"
sensitivity_df.to_csv(save_path, index=False)

print("=== CAPACITY SENSITIVITY ===")
display(sensitivity_df.round(3))
print("Saved:", save_path)


=== CAPACITY SENSITIVITY ===


,Safety_factor,Snapshots,Mean_hosts,Median_hosts,P95_hosts,Mean_CPU_util_pct_of_effective_capacity,Mean_memory_util_pct_of_effective_capacity,Snapshots_with_unplaceable_VMs_pct,Total_unplaceable_VM_snapshot_events
0,0.6,108,15.352,13.0,28.65,80.353,90.569,1.852,2
1,0.7,108,13.287,12.0,24.65,79.733,89.917,1.852,2
2,0.8,108,11.639,10.0,21.65,79.285,89.604,0.926,1
3,0.9,108,10.398,9.0,19.65,78.764,89.033,0.926,1


Saved: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/consolidation/capacity_sensitivity.csv


In [ ]:

# ================================================================
# FORECAST-DRIVEN VM CONSOLIDATION vs ORACLE FFD
# One self-contained evaluation cell. NO model retraining.
# ================================================================

import os
import glob
import warnings
import numpy as np
import pandas as pd
import torch
import joblib
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings("ignore")

# ----------------------- CONFIGURATION ----------------------------

ROOT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
TEST_DIR = ROOT / "data" / "splits" / "test"
OUT_DIR = ROOT / "results" / "consolidation_forecast"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "CPU usage [%]",
    "Memory usage [KB]",
    "Disk read throughput [KB/s]",
    "Disk write throughput [KB/s]",
]

LOOKBACK = 96                 # 8 hours of history
HORIZON_STEPS = 6             # 30 minutes ahead (6 x 5 minutes)
MODEL_HORIZON_INDEX = HORIZON_STEPS - 1
STEP_MINUTES = 5
SNAPSHOT_FREQ = "1h"
MIN_VMS_PER_SNAPSHOT = 20

HOST_CPU_RAW = 800.0          # Assumed 8 cores x 100% per core
HOST_MEMORY_RAW_KB = 64 * 1024 * 1024
SAFETY_FACTOR = 0.80

CPU_CAP = HOST_CPU_RAW * SAFETY_FACTOR
MEM_CAP = HOST_MEMORY_RAW_KB * SAFETY_FACTOR

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Reuse objects created by your training/evaluation notebook.
if "trained_models" not in globals():
    raise RuntimeError(
        "trained_models is missing. Run the existing model-loading cell first; "
        "do NOT retrain the models."
    )

if "scaler" not in globals():
    scaler_path = ROOT / "data" / "model_data" / "train_only_scaler.pkl"
    if not scaler_path.exists():
        raise FileNotFoundError(f"Training-only scaler not found: {scaler_path}")
    scaler = joblib.load(scaler_path)

MODEL_NAMES = ["PatchTST", "PatchTST_RevIN"]
for name in MODEL_NAMES:
    if name not in trained_models:
        raise RuntimeError(
            f"{name} is missing from trained_models. "
            f"Available models: {list(trained_models.keys())}"
        )

print("Device:", DEVICE)
print("Test directory:", TEST_DIR)
print("Effective host capacities:",
      f"CPU={CPU_CAP:.1f}, Memory={MEM_CAP:,.0f} KB")

# ----------------------- LOAD TEST TRACES -------------------------

files = sorted(glob.glob(str(TEST_DIR / "*.csv")))
if not files:
    raise FileNotFoundError(f"No test CSV files found in {TEST_DIR}")

traces = {}
all_starts, all_ends = [], []

for file in files:
    try:
        df = pd.read_csv(file)
        required = ["Timestamp", "VM_ID"] + FEATURES
        missing = [c for c in required if c not in df.columns]
        if missing:
            print(f"Skipping {Path(file).name}: missing {missing}")
            continue

        df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
        for col in FEATURES:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        df = df.dropna(subset=["Timestamp", "VM_ID"])
        df = df.sort_values("Timestamp")
        df = df.drop_duplicates(subset=["Timestamp"], keep="last")

        if len(df) < LOOKBACK + HORIZON_STEPS:
            continue

        # Keep NaN resource rows: they must invalidate a window,
        # not silently compress the timeline.
        idx = pd.DatetimeIndex(df["Timestamp"])
        values = df[FEATURES].to_numpy(dtype=np.float32)

        vm_id = str(df["VM_ID"].iloc[0])
        traces[vm_id] = {"index": idx, "values": values}

        all_starts.append(idx.min())
        all_ends.append(idx.max())

    except Exception as e:
        print(f"Skipping {Path(file).name}: {e}")

if len(traces) == 0:
    raise RuntimeError("No usable VM traces were loaded.")

print(f"Usable VM traces: {len(traces)}")

# Hourly wall-clock timestamps avoid different per-VM sampling phases.
global_start = min(all_starts).ceil(SNAPSHOT_FREQ)
global_end = max(all_ends).floor(SNAPSHOT_FREQ)
target_times = pd.date_range(global_start, global_end, freq=SNAPSHOT_FREQ)

if len(target_times) == 0:
    raise RuntimeError("No hourly target timestamps fall within the test period.")

print(f"Candidate hourly snapshots: {len(target_times)}")
print(f"Target period: {target_times.min()} to {target_times.max()}")

# ----------------------- MODEL PREPARATION -----------------------

models = {}
for name in MODEL_NAMES:
    model = trained_models[name].to(DEVICE)
    model.eval()
    models[name] = model

def predict_at_horizon(model, x_scaled):
    """Return inverse-scaled predictions at the 30-minute horizon."""
    x_tensor = torch.from_numpy(x_scaled).float().to(DEVICE)

    with torch.no_grad():
        output = model(x_tensor)

    if isinstance(output, (tuple, list)):
        output = output[0]
    if isinstance(output, dict):
        # Support common model output dictionaries.
        output = output.get("prediction", output.get("pred", None))
        if output is None:
            raise RuntimeError("Model returned a dict without prediction/pred.")

    if output.ndim != 3:
        raise RuntimeError(f"Expected 3D model output, got {tuple(output.shape)}")

    # Normalize layout to [batch, horizon, features].
    if output.shape[1] == len(FEATURES) and output.shape[2] >= HORIZON_STEPS:
        output = output.transpose(1, 2)
    elif output.shape[2] != len(FEATURES):
        raise RuntimeError(
            f"Unexpected model output shape: {tuple(output.shape)}"
        )

    if output.shape[1] < HORIZON_STEPS:
        raise RuntimeError(
            f"Model predicts only {output.shape[1]} steps; "
            f"{HORIZON_STEPS} are required."
        )

    pred_scaled = output[:, MODEL_HORIZON_INDEX, :].detach().cpu().numpy()
    return scaler.inverse_transform(pred_scaled)

# ----------------------- FFD PACKING -----------------------------

def ffd_assign(frame, cpu_col, mem_col):
    """
    First-Fit Decreasing assignment based on the specified demand columns.
    Returns host assignments and the predicted/oracle placement host count.
    """
    if len(frame) == 0:
        return {}, []

    work = frame.copy()
    cpu = np.maximum(
        np.nan_to_num(work[cpu_col].to_numpy(dtype=float), nan=0.0), 0.0
    )
    mem = np.maximum(
        np.nan_to_num(work[mem_col].to_numpy(dtype=float), nan=0.0), 0.0
    )

    # Sort by the largest normalized resource demand.
    dominant = np.maximum(cpu / CPU_CAP, mem / MEM_CAP)
    order = np.argsort(-dominant, kind="stable")

    host_cpu, host_mem = [], []
    assignments = {}

    for pos in order:
        vm_id = str(work.iloc[pos]["VM_ID"])
        c, m = float(cpu[pos]), float(mem[pos])

        placed = False
        for host_id in range(len(host_cpu)):
            if host_cpu[host_id] + c <= CPU_CAP and host_mem[host_id] + m <= MEM_CAP:
                host_cpu[host_id] += c
                host_mem[host_id] += m
                assignments[vm_id] = host_id
                placed = True
                break

        if not placed:
            host_id = len(host_cpu)
            host_cpu.append(c)
            host_mem.append(m)
            assignments[vm_id] = host_id

    host_loads = [
        {"host_id": i, "cpu": host_cpu[i], "memory": host_mem[i]}
        for i in range(len(host_cpu))
    ]
    return assignments, host_loads

def evaluate_assignment(frame, assignments):
    """Evaluate actual demands on the hosts chosen by a placement strategy."""
    host_actual = defaultdict(lambda: {"cpu": 0.0, "memory": 0.0})

    for _, row in frame.iterrows():
        host_id = assignments[str(row["VM_ID"])]
        host_actual[host_id]["cpu"] += max(float(row["actual_cpu"]), 0.0)
        host_actual[host_id]["memory"] += max(float(row["actual_memory"]), 0.0)

    cpu_over = []
    mem_over = []
    cpu_over_amount = []
    mem_over_amount = []

    for loads in host_actual.values():
        c_over = max(loads["cpu"] - CPU_CAP, 0.0)
        m_over = max(loads["memory"] - MEM_CAP, 0.0)
        cpu_over.append(c_over > 0)
        mem_over.append(m_over > 0)
        cpu_over_amount.append(c_over)
        mem_over_amount.append(m_over)

    n_hosts = len(host_actual)
    return {
        "hosts_used": n_hosts,
        "actual_cpu_overloaded_hosts": int(sum(cpu_over)),
        "actual_memory_overloaded_hosts": int(sum(mem_over)),
        "actual_either_resource_overloaded_hosts": int(
            sum(c or m for c, m in zip(cpu_over, mem_over))
        ),
        "actual_cpu_overload_total": float(sum(cpu_over_amount)),
        "actual_memory_overload_total_kb": float(sum(mem_over_amount)),
        "actual_cpu_util_pct_of_effective_capacity": (
            100.0 * sum(v["cpu"] for v in host_actual.values()) /
            (n_hosts * CPU_CAP) if n_hosts else np.nan
        ),
        "actual_memory_util_pct_of_effective_capacity": (
            100.0 * sum(v["memory"] for v in host_actual.values()) /
            (n_hosts * MEM_CAP) if n_hosts else np.nan
        ),
    }

# ----------------------- RUN SNAPSHOT SIMULATION -----------------

snapshot_rows = []
placement_rows = []
skipped_no_window = 0
skipped_small_snapshot = 0
valid_window_count = 0

for snap_no, target_time in enumerate(target_times, start=1):
    records = []
    windows = []
    vm_ids = []

    for vm_id, trace in traces.items():
        idx = trace["index"]
        vals = trace["values"]

        # Use pandas Timestamp lookup, avoiding numpy datetime-unit key mismatch.
        target_idx = idx.get_indexer([target_time])[0]
        if target_idx < 0:
            continue

        # For a target at i, the input window ends 30 minutes earlier.
        input_end = target_idx - HORIZON_STEPS
        input_start = input_end - LOOKBACK + 1
        continuity_start = input_start

        if input_start < 0:
            continue

        # Verify the complete lookback-to-target interval has exact 5-min cadence.
        interval_idx = idx[continuity_start:target_idx + 1]
        expected_len = LOOKBACK + HORIZON_STEPS
        if len(interval_idx) != expected_len:
            continue

        diffs = np.diff(interval_idx.asi8)
        expected_ns = STEP_MINUTES * 60 * 1_000_000_000
        if len(diffs) != expected_len - 1 or not np.all(diffs == expected_ns):
            continue

        x = vals[input_start:input_end + 1, :]
        actual = vals[target_idx, :]

        if x.shape != (LOOKBACK, len(FEATURES)):
            continue
        if not np.isfinite(x).all() or not np.isfinite(actual).all():
            continue

        # Negative values are invalid for resource demand.
        if np.any(x < 0) or np.any(actual < 0):
            continue

        windows.append(x)
        vm_ids.append(vm_id)
        records.append({
            "Timestamp": target_time,
            "VM_ID": vm_id,
            "actual_cpu": float(actual[0]),
            "actual_memory": float(actual[1]),
        })

    if len(records) < MIN_VMS_PER_SNAPSHOT:
        skipped_small_snapshot += 1
        continue

    X_raw = np.stack(windows).astype(np.float32)
    original_shape = X_raw.shape
    X_scaled = scaler.transform(X_raw.reshape(-1, len(FEATURES)))
    X_scaled = X_scaled.reshape(original_shape).astype(np.float32)

    frame = pd.DataFrame(records)
    valid_window_count += len(frame)

    # Compare both trained models independently against the same oracle snapshot.
    for model_name, model in models.items():
        pred = predict_at_horizon(model, X_scaled)

        frame_model = frame.copy()
        frame_model["pred_cpu"] = np.maximum(pred[:, 0], 0.0)
        frame_model["pred_memory"] = np.maximum(pred[:, 1], 0.0)

        # Oracle: placement knows actual instantaneous CPU and memory demand.
        oracle_assignment, _ = ffd_assign(
            frame_model, "actual_cpu", "actual_memory"
        )
        oracle_metrics = evaluate_assignment(frame_model, oracle_assignment)

        # Forecast: placement uses only the model's predicted CPU and memory.
        forecast_assignment, forecast_host_loads = ffd_assign(
            frame_model, "pred_cpu", "pred_memory"
        )
        forecast_metrics = evaluate_assignment(
            frame_model, forecast_assignment
        )

        snapshot_rows.append({
            "Timestamp": target_time,
            "Model": model_name,
            "VM_count": len(frame_model),
            "oracle_hosts": oracle_metrics["hosts_used"],
            "forecast_hosts": forecast_metrics["hosts_used"],
            "extra_hosts_vs_oracle": (
                forecast_metrics["hosts_used"] - oracle_metrics["hosts_used"]
            ),
            "oracle_actual_overloaded_hosts":
                oracle_metrics["actual_either_resource_overloaded_hosts"],
            "forecast_actual_overloaded_hosts":
                forecast_metrics["actual_either_resource_overloaded_hosts"],
            "forecast_actual_cpu_overloaded_hosts":
                forecast_metrics["actual_cpu_overloaded_hosts"],
            "forecast_actual_memory_overloaded_hosts":
                forecast_metrics["actual_memory_overloaded_hosts"],
            "forecast_actual_cpu_overload_total":
                forecast_metrics["actual_cpu_overload_total"],
            "forecast_actual_memory_overload_total_kb":
                forecast_metrics["actual_memory_overload_total_kb"],
            "forecast_actual_cpu_util_pct":
                forecast_metrics["actual_cpu_util_pct_of_effective_capacity"],
            "forecast_actual_memory_util_pct":
                forecast_metrics["actual_memory_util_pct_of_effective_capacity"],
            "oracle_actual_cpu_util_pct":
                oracle_metrics["actual_cpu_util_pct_of_effective_capacity"],
            "oracle_actual_memory_util_pct":
                oracle_metrics["actual_memory_util_pct_of_effective_capacity"],
        })

        for vm_id in vm_ids:
            placement_rows.append({
                "Timestamp": target_time,
                "Model": model_name,
                "VM_ID": vm_id,
                "actual_cpu": float(
                    frame_model.loc[frame_model["VM_ID"] == vm_id, "actual_cpu"].iloc[0]
                ),
                "actual_memory": float(
                    frame_model.loc[frame_model["VM_ID"] == vm_id, "actual_memory"].iloc[0]
                ),
                "pred_cpu": float(
                    frame_model.loc[frame_model["VM_ID"] == vm_id, "pred_cpu"].iloc[0]
                ),
                "pred_memory": float(
                    frame_model.loc[frame_model["VM_ID"] == vm_id, "pred_memory"].iloc[0]
                ),
                "forecast_host_id": forecast_assignment[vm_id],
                "oracle_host_id": oracle_assignment[vm_id],
            })

    if snap_no % 24 == 0:
        print(
            f"Checked {snap_no}/{len(target_times)} candidate timestamps; "
            f"successful snapshots so far: "
            f"{len(snapshot_rows) // len(MODEL_NAMES)}"
        )

# ----------------------- SAVE AND SUMMARIZE ----------------------

if not snapshot_rows:
    raise RuntimeError(
        "No snapshots passed validation. Check Timestamp cadence, test split "
        "coverage, feature columns, and scaler/model compatibility. "
        f"Traces loaded={len(traces)}, candidate times={len(target_times)}, "
        f"too-small snapshots={skipped_small_snapshot}."
    )

snapshot_df = pd.DataFrame(snapshot_rows)
placement_df = pd.DataFrame(placement_rows)

snapshot_path = OUT_DIR / "forecast_vs_oracle_snapshots.csv"
placement_path = OUT_DIR / "forecast_vs_oracle_placements.csv"
summary_path = OUT_DIR / "forecast_vs_oracle_summary.csv"

snapshot_df.to_csv(snapshot_path, index=False)
placement_df.to_csv(placement_path, index=False)

summary = (
    snapshot_df.groupby("Model")
    .agg(
        snapshots=("Timestamp", "nunique"),
        mean_vms=("VM_count", "mean"),
        mean_oracle_hosts=("oracle_hosts", "mean"),
        mean_forecast_hosts=("forecast_hosts", "mean"),
        mean_extra_hosts_vs_oracle=("extra_hosts_vs_oracle", "mean"),
        p95_extra_hosts_vs_oracle=(
            "extra_hosts_vs_oracle", lambda s: float(np.percentile(s, 95))
        ),
        mean_forecast_actual_overloaded_hosts=(
            "forecast_actual_overloaded_hosts", "mean"
        ),
        snapshots_with_forecast_overload=(
            "forecast_actual_overloaded_hosts", lambda s: int((s > 0).sum())
        ),
        mean_forecast_actual_cpu_util_pct=("forecast_actual_cpu_util_pct", "mean"),
        mean_forecast_actual_memory_util_pct=(
            "forecast_actual_memory_util_pct", "mean"
        ),
        mean_oracle_actual_overloaded_hosts=(
            "oracle_actual_overloaded_hosts", "mean"
        ),
    )
    .reset_index()
)

summary.to_csv(summary_path, index=False)

print("\n================ EVALUATION COMPLETE ================")
print(f"Usable VMs: {len(traces)}")
print(f"Candidate hourly timestamps: {len(target_times)}")
print(f"Valid VM-target windows: {valid_window_count}")
print(f"Skipped snapshots with fewer than {MIN_VMS_PER_SNAPSHOT} VMs: "
      f"{skipped_small_snapshot}")
print("\nSUMMARY — forecast-driven placement evaluated using ACTUAL demand:")
display(summary.round(3))

print("\nSaved files:")
print(snapshot_path)
print(placement_path)
print(summary_path)


Device: cuda
Test directory: /content/drive/MyDrive/Bitbrains_Fresh_Research/data/splits/test
Effective host capacities: CPU=640.0, Memory=53,687,091 KB
Usable VM traces: 1217
Candidate hourly snapshots: 540
Target period: 2013-08-20 02:00:00 to 2013-09-11 13:00:00
Checked 456/540 candidate timestamps; successful snapshots so far: 15
Checked 480/540 candidate timestamps; successful snapshots so far: 39
Checked 504/540 candidate timestamps; successful snapshots so far: 63
Checked 528/540 candidate timestamps; successful snapshots so far: 87

================ EVALUATION COMPLETE ================
Usable VMs: 1217
Candidate hourly timestamps: 540
Valid VM-target windows: 114141
Skipped snapshots with fewer than 20 VMs: 441

SUMMARY — forecast-driven placement evaluated using ACTUAL demand:


,Model,snapshots,mean_vms,mean_oracle_hosts,mean_forecast_hosts,mean_extra_hosts_vs_oracle,p95_extra_hosts_vs_oracle,mean_forecast_actual_overloaded_hosts,snapshots_with_forecast_overload,mean_forecast_actual_cpu_util_pct,mean_forecast_actual_memory_util_pct,mean_oracle_actual_overloaded_hosts
0,PatchTST,99,1152.939,12.525,10.384,-2.141,3.0,6.798,99,92.657,112.715,0.0
1,PatchTST_RevIN,99,1152.939,12.525,10.929,-1.596,3.1,6.525,99,87.970,107.182,0.0



Saved files:
/content/drive/MyDrive/Bitbrains_Fresh_Research/results/consolidation_forecast/forecast_vs_oracle_snapshots.csv
/content/drive/MyDrive/Bitbrains_Fresh_Research/results/consolidation_forecast/forecast_vs_oracle_placements.csv
/content/drive/MyDrive/Bitbrains_Fresh_Research/results/consolidation_forecast/forecast_vs_oracle_summary.csv


In [ ]:

# ================================================================
# VALIDATION-CALIBRATED FORECAST PLACEMENT
# Validation -> learn margins; Test -> evaluate once.
# Reuses: traces, snapshot_df, trained_models, scaler, DEVICE
# ================================================================

import numpy as np
import pandas as pd
import torch
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
VAL_DIR = ROOT / "data" / "splits" / "val"
OUT_DIR = ROOT / "results" / "consolidation_forecast"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "CPU usage [%]",
    "Memory usage [KB]",
    "Disk read throughput [KB/s]",
    "Disk write throughput [KB/s]",
]

LOOKBACK = 96
HORIZON_STEPS = 6
HORIZON_INDEX = HORIZON_STEPS - 1
STEP_NS = 5 * 60 * 1_000_000_000
CPU_CAP = 800.0 * 0.80
MEM_CAP = 64 * 1024 * 1024 * 0.80
MIN_VMS = 20

# Quantile is fixed in advance; do not select it based on test results.
RESIDUAL_QUANTILE = 0.95
MODEL_NAMES = ["PatchTST", "PatchTST_RevIN"]

assert "traces" in globals(), "Run the successful consolidation cell first."
assert "snapshot_df" in globals(), "snapshot_df is missing; run the prior cell first."
assert "trained_models" in globals() and "scaler" in globals()

# ---------- Build validation traces without dropping time rows ----------

import glob
val_traces = {}

for fp in sorted(glob.glob(str(VAL_DIR / "*.csv"))):
    try:
        d = pd.read_csv(fp)
        if not set(["Timestamp", "VM_ID"] + FEATURES).issubset(d.columns):
            continue
        d["Timestamp"] = pd.to_datetime(d["Timestamp"], errors="coerce")
        for c in FEATURES:
            d[c] = pd.to_numeric(d[c], errors="coerce")
        d = d.dropna(subset=["Timestamp", "VM_ID"]).sort_values("Timestamp")
        d = d.drop_duplicates("Timestamp", keep="last")
        if len(d) < LOOKBACK + HORIZON_STEPS:
            continue
        vm = str(d["VM_ID"].iloc[0])
        val_traces[vm] = {
            "index": pd.DatetimeIndex(d["Timestamp"]),
            "values": d[FEATURES].to_numpy(dtype=np.float32)
        }
    except Exception as e:
        print("Skipping validation file:", fp, "|", e)

print("Validation VM traces:", len(val_traces))
if not val_traces:
    raise RuntimeError("No validation traces loaded.")

# ---------- Predict validation windows at 30-minute horizon ----------

def predict_horizon(model, x_scaled):
    model.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        out = model(torch.as_tensor(x_scaled, dtype=torch.float32, device=device))
    if isinstance(out, (tuple, list)):
        out = out[0]
    if isinstance(out, dict):
        out = out.get("prediction", out.get("pred"))
    if out is None or out.ndim != 3:
        raise RuntimeError("Unexpected model output.")

    if out.shape[1] == len(FEATURES) and out.shape[2] >= HORIZON_STEPS:
        out = out.transpose(1, 2)
    if out.shape[2] != len(FEATURES) or out.shape[1] < HORIZON_STEPS:
        raise RuntimeError(f"Unexpected output shape: {tuple(out.shape)}")

    p = out[:, HORIZON_INDEX, :].detach().cpu().numpy()
    return scaler.inverse_transform(p)

validation_predictions = {m: [] for m in MODEL_NAMES}
validation_actuals = []

# Use a deterministic sample of windows per VM to bound runtime.
MAX_VAL_WINDOWS_PER_VM = 120
for vm, tr in val_traces.items():
    idx, vals = tr["index"], tr["values"]
    candidates = []

    # Target index i; input ends 30 minutes before i.
    for i in range(LOOKBACK + HORIZON_STEPS - 1, len(idx), 6):
        s = i - HORIZON_STEPS - LOOKBACK + 1
        e = i - HORIZON_STEPS + 1
        if s < 0:
            continue
        if len(idx[s:i+1]) != LOOKBACK + HORIZON_STEPS:
            continue
        if not np.all(np.diff(idx[s:i+1].asi8) == STEP_NS):
            continue
        x = vals[s:e]
        y = vals[i]
        if x.shape != (LOOKBACK, len(FEATURES)):
            continue
        if not np.isfinite(x).all() or not np.isfinite(y).all():
            continue
        if (x < 0).any() or (y < 0).any():
            continue
        candidates.append((x, y))

    if not candidates:
        continue

    if len(candidates) > MAX_VAL_WINDOWS_PER_VM:
        chosen = np.linspace(
            0, len(candidates)-1, MAX_VAL_WINDOWS_PER_VM
        ).astype(int)
        candidates = [candidates[j] for j in chosen]

    X = np.stack([c[0] for c in candidates]).astype(np.float32)
    Y = np.stack([c[1] for c in candidates]).astype(np.float32)
    Xs = scaler.transform(X.reshape(-1, len(FEATURES)))
    Xs = Xs.reshape(X.shape).astype(np.float32)

    validation_actuals.append(Y)
    for model_name in MODEL_NAMES:
        P = predict_horizon(trained_models[model_name], Xs)
        validation_predictions[model_name].append(P)

if not validation_actuals:
    raise RuntimeError("No valid validation windows. Check validation timestamps.")

Y_val = np.concatenate(validation_actuals, axis=0)
print("Validation windows used:", len(Y_val))

# Positive residual = actual demand exceeds prediction.
# Per-resource one-sided 95th-percentile residual margin.
margins = {}
margin_rows = []

for model_name in MODEL_NAMES:
    P_val = np.concatenate(validation_predictions[model_name], axis=0)
    residual = Y_val - P_val

    margin = np.maximum(
        np.quantile(residual, RESIDUAL_QUANTILE, axis=0), 0.0
    )
    margins[model_name] = margin

    for j, feature in enumerate(FEATURES):
        margin_rows.append({
            "Model": model_name,
            "Resource": feature,
            "Validation_residual_q": RESIDUAL_QUANTILE,
            "Safety_margin": float(margin[j]),
            "Validation_MAE": float(np.mean(np.abs(P_val[:, j] - Y_val[:, j]))),
            "Validation_underprediction_rate": float(
                np.mean(P_val[:, j] < Y_val[:, j])
            ),
        })

margin_df = pd.DataFrame(margin_rows)
margin_df.to_csv(OUT_DIR / "validation_resource_safety_margins.csv", index=False)

print("\nMargins learned from validation only:")
display(margin_df.round(4))

# ---------- FFD placement and actual-demand evaluation ----------

def ffd(frame, cpu_col, mem_col):
    cpu = np.maximum(frame[cpu_col].to_numpy(float), 0)
    mem = np.maximum(frame[mem_col].to_numpy(float), 0)
    dominant = np.maximum(cpu / CPU_CAP, mem / MEM_CAP)
    order = np.argsort(-dominant, kind="stable")

    host_cpu, host_mem, assignment = [], [], {}
    for pos in order:
        vm = str(frame.iloc[pos]["VM_ID"])
        c, m = cpu[pos], mem[pos]
        host = None
        for h in range(len(host_cpu)):
            if host_cpu[h] + c <= CPU_CAP and host_mem[h] + m <= MEM_CAP:
                host = h
                break
        if host is None:
            host = len(host_cpu)
            host_cpu.append(0.0)
            host_mem.append(0.0)
        host_cpu[host] += c
        host_mem[host] += m
        assignment[vm] = host
    return assignment

def assess_actual(frame, assignment):
    loads = {}
    for _, r in frame.iterrows():
        h = assignment[str(r["VM_ID"])]
        if h not in loads:
            loads[h] = [0.0, 0.0]
        loads[h][0] += max(float(r["actual_cpu"]), 0.0)
        loads[h][1] += max(float(r["actual_memory"]), 0.0)

    n = len(loads)
    cpu_bad = sum(c > CPU_CAP for c, m in loads.values())
    mem_bad = sum(m > MEM_CAP for c, m in loads.values())
    either_bad = sum(c > CPU_CAP or m > MEM_CAP for c, m in loads.values())

    return {
        "hosts": n,
        "actual_cpu_overloaded_hosts": int(cpu_bad),
        "actual_memory_overloaded_hosts": int(mem_bad),
        "actual_either_overloaded_hosts": int(either_bad),
        "actual_cpu_util_pct": (
            100 * sum(v[0] for v in loads.values()) / (n * CPU_CAP)
            if n else np.nan
        ),
        "actual_memory_util_pct": (
            100 * sum(v[1] for v in loads.values()) / (n * MEM_CAP)
            if n else np.nan
        )
    }

# ---------- Re-evaluate exact test snapshots from the prior cell ----------
# Rebuild predictions from saved placement rows produced by the previous cell.
placement_path = OUT_DIR / "forecast_vs_oracle_placements.csv"
if not placement_path.exists():
    raise FileNotFoundError(
        f"Missing {placement_path}. Run the prior successful test simulation first."
    )

placements = pd.read_csv(placement_path, parse_dates=["Timestamp"])
required_placement_cols = {
    "Timestamp", "Model", "VM_ID", "actual_cpu", "actual_memory",
    "pred_cpu", "pred_memory"
}
if not required_placement_cols.issubset(placements.columns):
    raise RuntimeError("Previous placement CSV does not have expected columns.")

calibrated_rows = []
for (ts, model_name), frame in placements.groupby(["Timestamp", "Model"]):
    if model_name not in margins or len(frame) < MIN_VMS:
        continue

    frame = frame.copy()
    margin = margins[model_name]

    # Clamp negative forecasts and apply validation-derived one-sided margins.
    frame["cal_cpu"] = np.maximum(frame["pred_cpu"].to_numpy(float) + margin[0], 0)
    frame["cal_memory"] = np.maximum(
        frame["pred_memory"].to_numpy(float) + margin[1], 0
    )

    # Baseline raw forecast placement.
    raw_assignment = ffd(frame, "pred_cpu", "pred_memory")
    raw_metrics = assess_actual(frame, raw_assignment)

    # Calibrated forecast placement.
    cal_assignment = ffd(frame, "cal_cpu", "cal_memory")
    cal_metrics = assess_actual(frame, cal_assignment)

    # Oracle placement for the same VM set.
    oracle_assignment = ffd(frame, "actual_cpu", "actual_memory")
    oracle_metrics = assess_actual(frame, oracle_assignment)

    calibrated_rows.append({
        "Timestamp": ts,
        "Model": model_name,
        "VM_count": len(frame),
        "oracle_hosts": oracle_metrics["hosts"],
        "raw_forecast_hosts": raw_metrics["hosts"],
        "calibrated_forecast_hosts": cal_metrics["hosts"],
        "raw_actual_overloaded_hosts": raw_metrics["actual_either_overloaded_hosts"],
        "calibrated_actual_overloaded_hosts": cal_metrics["actual_either_overloaded_hosts"],
        "calibrated_cpu_overloaded_hosts": cal_metrics["actual_cpu_overloaded_hosts"],
        "calibrated_memory_overloaded_hosts": cal_metrics["actual_memory_overloaded_hosts"],
        "calibrated_actual_cpu_util_pct": cal_metrics["actual_cpu_util_pct"],
        "calibrated_actual_memory_util_pct": cal_metrics["actual_memory_util_pct"],
        "oracle_actual_overloaded_hosts": oracle_metrics["actual_either_overloaded_hosts"],
    })

cal_df = pd.DataFrame(calibrated_rows)
if cal_df.empty:
    raise RuntimeError("No matching test snapshots found for calibration evaluation.")

cal_df.to_csv(OUT_DIR / "validation_calibrated_test_snapshots.csv", index=False)

summary = (
    cal_df.groupby("Model")
    .agg(
        snapshots=("Timestamp", "nunique"),
        mean_vms=("VM_count", "mean"),
        oracle_mean_hosts=("oracle_hosts", "mean"),
        raw_mean_hosts=("raw_forecast_hosts", "mean"),
        calibrated_mean_hosts=("calibrated_forecast_hosts", "mean"),
        raw_mean_overloaded_hosts=("raw_actual_overloaded_hosts", "mean"),
        calibrated_mean_overloaded_hosts=("calibrated_actual_overloaded_hosts", "mean"),
        snapshots_with_calibrated_overload=(
            "calibrated_actual_overloaded_hosts", lambda s: int((s > 0).sum())
        ),
        calibrated_mean_cpu_util_pct=("calibrated_actual_cpu_util_pct", "mean"),
        calibrated_mean_memory_util_pct=("calibrated_actual_memory_util_pct", "mean"),
    )
    .reset_index()
)

summary.to_csv(OUT_DIR / "validation_calibrated_test_summary.csv", index=False)

print("\n========== VALIDATION-CALIBRATED TEST RESULTS ==========")
display(summary.round(3))
print("\nSaved:")
print(OUT_DIR / "validation_resource_safety_margins.csv")
print(OUT_DIR / "validation_calibrated_test_snapshots.csv")
print(OUT_DIR / "validation_calibrated_test_summary.csv")


Validation VM traces: 1217
Validation windows used: 144089

Margins learned from validation only:


,Model,Resource,Validation_residual_q,Safety_margin,Validation_MAE,Validation_underprediction_rate
0,PatchTST,CPU usage [%],0.95,3.7727,1.6123,0.8801
1,PatchTST,Memory usage [KB],0.95,441314.5000,235610.0781,0.7134
2,PatchTST,Disk read throughput [KB/s],0.95,84.6722,222.1559,0.9746
3,PatchTST,Disk write throughput [KB/s],0.95,25.2821,64.4699,0.9473
4,PatchTST_RevIN,CPU usage [%],0.95,1.9746,1.3715,0.4004
5,PatchTST_RevIN,Memory usage [KB],0.95,415301.9062,230471.0312,0.4587
6,PatchTST_RevIN,Disk read throughput [KB/s],0.95,3.1500,142.8777,0.1630
7,PatchTST_RevIN,Disk write throughput [KB/s],0.95,9.9424,50.2712,0.3619



========== VALIDATION-CALIBRATED TEST RESULTS ==========


,Model,snapshots,mean_vms,oracle_mean_hosts,raw_mean_hosts,calibrated_mean_hosts,raw_mean_overloaded_hosts,calibrated_mean_overloaded_hosts,snapshots_with_calibrated_overload,calibrated_mean_cpu_util_pct,calibrated_mean_memory_util_pct
0,PatchTST,99,1152.939,12.525,10.384,20.071,6.798,3.737,78,45.361,55.886
1,PatchTST_RevIN,99,1152.939,12.525,10.929,19.869,6.525,3.455,72,45.963,56.641



Saved:
/content/drive/MyDrive/Bitbrains_Fresh_Research/results/consolidation_forecast/validation_resource_safety_margins.csv
/content/drive/MyDrive/Bitbrains_Fresh_Research/results/consolidation_forecast/validation_calibrated_test_snapshots.csv
/content/drive/MyDrive/Bitbrains_Fresh_Research/results/consolidation_forecast/validation_calibrated_test_summary.csv


In [ ]:

import pandas as pd
from pathlib import Path

out = Path(
    "/content/drive/MyDrive/Bitbrains_Fresh_Research/results/consolidation_forecast"
)

p = out / "validation_calibrated_test_summary.csv"
if not p.exists():
    raise FileNotFoundError(
        f"Cannot find {p}. The calibration evaluation may not have completed."
    )

summary = pd.read_csv(p)

cols = [
    "Model",
    "snapshots",
    "mean_vms",
    "oracle_mean_hosts",
    "raw_mean_hosts",
    "calibrated_mean_hosts",
    "raw_mean_overloaded_hosts",
    "calibrated_mean_overloaded_hosts",
    "snapshots_with_calibrated_overload",
    "calibrated_mean_cpu_util_pct",
    "calibrated_mean_memory_util_pct",
]

display(summary[cols].round(3))

# Compare raw vs calibrated overloads and host counts
summary["overload_reduction"] = (
    summary["raw_mean_overloaded_hosts"]
    - summary["calibrated_mean_overloaded_hosts"]
)
summary["additional_hosts_after_calibration"] = (
    summary["calibrated_mean_hosts"] - summary["raw_mean_hosts"]
)

print("\nInterpretation:")
display(summary[[
    "Model",
    "overload_reduction",
    "additional_hosts_after_calibration",
    "snapshots_with_calibrated_overload",
]].round(3))


,Model,snapshots,mean_vms,oracle_mean_hosts,raw_mean_hosts,calibrated_mean_hosts,raw_mean_overloaded_hosts,calibrated_mean_overloaded_hosts,snapshots_with_calibrated_overload,calibrated_mean_cpu_util_pct,calibrated_mean_memory_util_pct
0,PatchTST,99,1152.939,12.525,10.384,20.071,6.798,3.737,78,45.361,55.886
1,PatchTST_RevIN,99,1152.939,12.525,10.929,19.869,6.525,3.455,72,45.963,56.641



Interpretation:


,Model,overload_reduction,additional_hosts_after_calibration,snapshots_with_calibrated_overload
0,PatchTST,3.061,9.687,78
1,PatchTST_RevIN,3.071,8.939,72


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

out = Path(
    "/content/drive/MyDrive/Bitbrains_Fresh_Research/results/consolidation_forecast"
)
p = out / "forecast_vs_oracle_placements.csv"
df = pd.read_csv(p, parse_dates=["Timestamp"])

CPU_CAP = 800.0 * 0.80
MEM_CAP = 64 * 1024 * 1024 * 0.80

rows = []

for (ts, model), g in df.groupby(["Timestamp", "Model"]):
    hosts = defaultdict(lambda: [0.0, 0.0])

    for _, r in g.iterrows():
        h = int(r["forecast_host_id"])
        hosts[h][0] += max(float(r["actual_cpu"]), 0.0)
        hosts[h][1] += max(float(r["actual_memory"]), 0.0)

    cpu_excess = [max(v[0] - CPU_CAP, 0.0) for v in hosts.values()]
    mem_excess = [max(v[1] - MEM_CAP, 0.0) for v in hosts.values()]

    rows.append({
        "Timestamp": ts,
        "Model": model,
        "hosts": len(hosts),
        "cpu_overloaded_hosts": sum(x > 0 for x in cpu_excess),
        "memory_overloaded_hosts": sum(x > 0 for x in mem_excess),
        "max_cpu_excess": max(cpu_excess, default=0.0),
        "max_memory_excess_kb": max(mem_excess, default=0.0),
        "total_cpu_excess": sum(cpu_excess),
        "total_memory_excess_kb": sum(mem_excess),
    })

severity = pd.DataFrame(rows)

report = severity.groupby("Model").agg(
    snapshots=("Timestamp", "nunique"),
    mean_hosts=("hosts", "mean"),
    snapshots_with_cpu_overload=(
        "cpu_overloaded_hosts", lambda x: int((x > 0).sum())
    ),
    snapshots_with_memory_overload=(
        "memory_overloaded_hosts", lambda x: int((x > 0).sum())
    ),
    mean_cpu_overloaded_hosts=("cpu_overloaded_hosts", "mean"),
    mean_memory_overloaded_hosts=("memory_overloaded_hosts", "mean"),
    p95_max_cpu_excess=("max_cpu_excess", lambda x: np.percentile(x, 95)),
    p95_max_memory_excess_kb=(
        "max_memory_excess_kb", lambda x: np.percentile(x, 95)
    ),
).reset_index()

display(report.round(3))
report.to_csv(out / "raw_overload_severity.csv", index=False)
print("Saved:", out / "raw_overload_severity.csv")


,Model,snapshots,mean_hosts,snapshots_with_cpu_overload,snapshots_with_memory_overload,mean_cpu_overloaded_hosts,mean_memory_overloaded_hosts,p95_max_cpu_excess,p95_max_memory_excess_kb
0,PatchTST,99,10.384,87,98,3.434,5.788,1200.348,1.976137e+08
1,PatchTST_RevIN,99,10.929,74,99,2.707,5.707,1126.950,1.348414e+08


Saved: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/consolidation_forecast/raw_overload_severity.csv


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
TEST_DIR = ROOT / "data" / "splits" / "test"
OUT_DIR = ROOT / "results" / "consolidation_forecast"

CPU_CAP = 800.0 * 0.80
MEM_CAP_KB = 64 * 1024 * 1024 * 0.80

FEATURES = ["CPU usage [%]", "Memory usage [KB]"]

# ---------- 1. Inspect actual per-VM resource demand ----------

rows = []
for fp in TEST_DIR.glob("*.csv"):
    d = pd.read_csv(fp, usecols=lambda c: c in ["VM_ID"] + FEATURES)
    if not all(c in d.columns for c in ["VM_ID"] + FEATURES):
        continue

    for c in FEATURES:
        d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d.dropna(subset=FEATURES)
    if d.empty:
        continue

    rows.append({
        "VM_ID": str(d["VM_ID"].iloc[0]),
        "rows": len(d),
        "mean_cpu": d[FEATURES[0]].mean(),
        "p95_cpu": d[FEATURES[0]].quantile(0.95),
        "max_cpu": d[FEATURES[0]].max(),
        "mean_memory_kb": d[FEATURES[1]].mean(),
        "p95_memory_kb": d[FEATURES[1]].quantile(0.95),
        "max_memory_kb": d[FEATURES[1]].max(),
    })

vm_audit = pd.DataFrame(rows)
if vm_audit.empty:
    raise RuntimeError("No VM demand data could be read.")

print("VMs audited:", len(vm_audit))
print("\nResource distribution across VM-level statistics:")
display(vm_audit[[
    "mean_cpu", "p95_cpu", "max_cpu",
    "mean_memory_kb", "p95_memory_kb", "max_memory_kb"
]].describe(percentiles=[.5, .9, .95, .99]).round(2))

# ---------- 2. Find VMs whose own demand exceeds assumed capacity ----------

cpu_bad = vm_audit[vm_audit["max_cpu"] > CPU_CAP]
mem_bad = vm_audit[vm_audit["max_memory_kb"] > MEM_CAP_KB]

print(f"\nAssumed effective CPU capacity: {CPU_CAP:.2f}")
print(f"VMs with peak CPU demand above capacity: {len(cpu_bad)}")
print(f"\nAssumed effective memory capacity: {MEM_CAP_KB:,.0f} KB")
print(f"VMs with peak memory demand above capacity: {len(mem_bad)}")

print("\nLargest individual VM memory peaks (GB, using 1 GiB = 1024^3 bytes):")
top_memory = vm_audit.nlargest(15, "max_memory_kb").copy()
top_memory["max_memory_GiB_if_KB"] = (
    top_memory["max_memory_kb"] * 1024 / (1024 ** 3)
)
display(top_memory[["VM_ID", "max_memory_kb", "max_memory_GiB_if_KB"]])

# ---------- 3. Check forecast and actual demand in saved placements ----------

placement_file = OUT_DIR / "forecast_vs_oracle_placements.csv"
if placement_file.exists():
    p = pd.read_csv(placement_file)

    for c in ["actual_cpu", "actual_memory", "pred_cpu", "pred_memory"]:
        p[c] = pd.to_numeric(p[c], errors="coerce")

    print("\nSaved placement row-level distributions:")
    display(p[[
        "actual_cpu", "pred_cpu", "actual_memory", "pred_memory"
    ]].describe(percentiles=[.5, .9, .95, .99]).round(2))

    print("\nRows individually exceeding the assumed effective host capacity:")
    print("Actual CPU:", int((p["actual_cpu"] > CPU_CAP).sum()))
    print("Actual memory:", int((p["actual_memory"] > MEM_CAP_KB).sum()))
    print("Predicted CPU:", int((p["pred_cpu"] > CPU_CAP).sum()))
    print("Predicted memory:", int((p["pred_memory"] > MEM_CAP_KB).sum()))

# Save audit
OUT_DIR.mkdir(parents=True, exist_ok=True)
vm_audit.to_csv(OUT_DIR / "vm_capacity_unit_audit.csv", index=False)
print("\nSaved:", OUT_DIR / "vm_capacity_unit_audit.csv")


VMs audited: 1217

Resource distribution across VM-level statistics:


,mean_cpu,p95_cpu,max_cpu,mean_memory_kb,p95_memory_kb,max_memory_kb
count,1217.00,1217.00,1217.00,1217.00,1217.00,1217.00
mean,4.85,12.09,21.75,498307.58,1457564.34,3803488.01
std,10.32,28.01,32.65,1152403.23,3780060.46,8673803.11
min,0.00,0.00,0.00,0.00,0.00,0.00
50%,1.24,1.50,5.73,133947.17,251656.60,752177.33
90%,16.40,44.17,94.77,1068206.82,2697774.08,9428794.08
95%,30.18,102.36,104.09,3482094.65,13907191.69,31948290.72
99%,46.76,104.35,105.29,5092813.94,17244112.82,37671558.45
max,82.55,105.23,126.47,14047796.71,18645077.20,82320205.07



Assumed effective CPU capacity: 640.00
VMs with peak CPU demand above capacity: 0

Assumed effective memory capacity: 53,687,091 KB
VMs with peak memory demand above capacity: 1

Largest individual VM memory peaks (GB, using 1 GiB = 1024^3 bytes):


,VM_ID,max_memory_kb,max_memory_GiB_if_KB
1142,145,8.232021e+07,78.506665
68,290,3.903499e+07,37.226665
36,258,3.875537e+07,36.959999
56,278,3.853167e+07,36.746665
28,250,3.844778e+07,36.666665
26,248,3.822409e+07,36.453332
18,240,3.816816e+07,36.399999
24,246,3.811224e+07,36.346665
66,288,3.811224e+07,36.346665
62,284,3.808428e+07,36.319999



Saved placement row-level distributions:


,actual_cpu,pred_cpu,actual_memory,pred_memory
count,228282.00,228282.00,228282.00,228282.00
mean,5.05,4.55,531371.16,446864.95
std,16.11,14.62,2001832.93,1548617.20
min,0.00,0.00,0.00,0.00
50%,1.00,0.89,89478.40,96160.23
90%,6.53,6.18,844451.44,772513.03
95%,25.27,20.75,1934970.38,1628959.03
99%,100.64,91.66,9842632.00,8539567.05
max,125.80,129.74,38447784.00,31259090.00



Rows individually exceeding the assumed effective host capacity:
Actual CPU: 0
Actual memory: 0
Predicted CPU: 0
Predicted memory: 0

Saved: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/consolidation_forecast/vm_capacity_unit_audit.csv


In [ ]:

# ================================================================
# NEW FORECASTING MODELS — CORRECT 4-FEATURE VERSION
# Existing loaders: [batch, 96, 4] -> [batch, 12, 4]
# New models ONLY: NLinear, TSMixer + persistence baseline
# Existing model checkpoints are not touched.
# ================================================================

import copy, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
LOOKBACK, HORIZON = 96, 12
MAX_EPOCHS, PATIENCE = 12, 4
LR, WEIGHT_DECAY = 3e-4, 1e-4

FEATURES_4 = [
    "CPU usage [%]",
    "Memory usage [KB]",
    "Disk read throughput [KB/s]",
    "Disk write throughput [KB/s]",
]
N_FEATURES = len(FEATURES_4)

ROOT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
OUT = ROOT / "results" / "new_forecasting_models"
CKPT = ROOT / "models" / "new_forecasting_models"
OUT.mkdir(parents=True, exist_ok=True)
CKPT.mkdir(parents=True, exist_ok=True)

for obj in ["train_loader", "val_loader", "test_loader", "scaler"]:
    if obj not in globals():
        raise RuntimeError(f"Missing {obj}; run the existing data preparation cells.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Validate actual loader dimensions before training.
sx, sy = next(iter(train_loader))
print("Device:", DEVICE)
print("Train X:", tuple(sx.shape), "| Train Y:", tuple(sy.shape))

if sx.shape[1:] != (LOOKBACK, N_FEATURES):
    raise RuntimeError(f"Unexpected input shape: {tuple(sx.shape)}")
if sy.shape[1:] != (HORIZON, N_FEATURES):
    raise RuntimeError(f"Unexpected target shape: {tuple(sy.shape)}")

# ---------------- NLinear ----------------

class NLinearNew(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(LOOKBACK, HORIZON)

    def forward(self, x):
        last = x[:, -1:, :]
        residual = x - last
        pred = self.linear(residual.transpose(1, 2))
        return pred.transpose(1, 2) + last

# ---------------- TSMixer ----------------

class MixerBlockNew(nn.Module):
    def __init__(self, hidden=128, dropout=0.1):
        super().__init__()
        self.norm_time = nn.LayerNorm(N_FEATURES)
        self.time_mlp = nn.Sequential(
            nn.Linear(LOOKBACK, LOOKBACK),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.norm_features = nn.LayerNorm(N_FEATURES)
        self.feature_mlp = nn.Sequential(
            nn.Linear(N_FEATURES, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, N_FEATURES),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        z = self.norm_time(x).transpose(1, 2)
        x = x + self.time_mlp(z).transpose(1, 2)
        x = x + self.feature_mlp(self.norm_features(x))
        return x

class TSMixerNew(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList([
            MixerBlockNew(),
            MixerBlockNew(),
        ])
        self.norm = nn.LayerNorm(N_FEATURES)
        self.head = nn.Linear(LOOKBACK, HORIZON)

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        z = self.norm(x).transpose(1, 2)
        return self.head(z).transpose(1, 2)

# ---------------- Training ----------------

def train_new_model(name, model):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=1
    )

    best_val = float("inf")
    best_state = None
    bad_epochs = 0
    history = []
    started = time.time()
    checkpoint = CKPT / f"{name}_best.pt"

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_sum, train_n = 0.0, 0

        for x, y in train_loader:
            x = x.to(DEVICE, dtype=torch.float32)
            y = y.to(DEVICE, dtype=torch.float32)

            optimizer.zero_grad(set_to_none=True)
            pred = model(x)
            loss = F.smooth_l1_loss(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_sum += loss.item() * len(x)
            train_n += len(x)

        model.eval()
        val_sum, val_n = 0.0, 0

        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(DEVICE, dtype=torch.float32)
                y = y.to(DEVICE, dtype=torch.float32)
                loss = F.smooth_l1_loss(model(x), y)
                val_sum += loss.item() * len(x)
                val_n += len(x)

        train_loss = train_sum / max(train_n, 1)
        val_loss = val_sum / max(val_n, 1)
        scheduler.step(val_loss)

        history.append({
            "epoch": epoch,
            "train_huber": train_loss,
            "val_huber": val_loss,
            "lr": optimizer.param_groups[0]["lr"],
        })

        print(
            f"{name} | epoch {epoch:02d}/{MAX_EPOCHS} | "
            f"train={train_loss:.6f} | val={val_loss:.6f}"
        )

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, checkpoint)
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("Early stopping:", name)
                break

    if best_state is None:
        raise RuntimeError(f"Training failed for {name}")

    model.load_state_dict(best_state)
    pd.DataFrame(history).to_csv(OUT / f"{name}_history.csv", index=False)
    return model, best_val, time.time() - started

# ---------------- Metrics in original units ----------------

def inverse_scale(a):
    shape = a.shape
    return scaler.inverse_transform(
        a.reshape(-1, N_FEATURES)
    ).reshape(shape)

def evaluate_new_model(name, model):
    model.eval()
    preds, actuals = [], []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE, dtype=torch.float32)
            p = model(x).cpu().numpy()
            if p.shape != tuple(y.shape):
                raise RuntimeError(
                    f"{name}: prediction shape {p.shape}, target {tuple(y.shape)}"
                )
            preds.append(p)
            actuals.append(y.numpy())

    ps = np.concatenate(preds)
    ys = np.concatenate(actuals)
    p = inverse_scale(ps)
    y = inverse_scale(ys)

    resource_rows, horizon_rows = [], []
    for j, feature in enumerate(FEATURES_4):
        yt, yp = y[:, :, j].ravel(), p[:, :, j].ravel()
        err = yp - yt
        ss = np.sum((yt - yt.mean()) ** 2)

        resource_rows.append({
            "Model": name,
            "Resource": feature,
            "MAE": np.mean(np.abs(err)),
            "RMSE": np.sqrt(np.mean(err**2)),
            "R2": 1 - np.sum(err**2) / ss if ss > 0 else np.nan,
            "WAPE_pct": 100 * np.sum(np.abs(err)) /
                        max(np.sum(np.abs(yt)), 1e-9),
            "sMAPE_pct": 100 * np.mean(
                2 * np.abs(err) / (np.abs(yt) + np.abs(yp) + 1e-8)
            ),
        })

        for step in [1, 3, 6, 12]:
            a = y[:, step - 1, j]
            b = p[:, step - 1, j]
            e = b - a
            ss_h = np.sum((a - a.mean()) ** 2)
            horizon_rows.append({
                "Model": name,
                "Resource": feature,
                "Horizon_minutes": step * 5,
                "MAE": np.mean(np.abs(e)),
                "RMSE": np.sqrt(np.mean(e**2)),
                "R2": 1 - np.sum(e**2) / ss_h if ss_h > 0 else np.nan,
            })

    se = ps - ys
    summary = {
        "Model": name,
        "Test_windows": len(ys),
        "Scaled_MAE": np.mean(np.abs(se)),
        "Scaled_RMSE": np.sqrt(np.mean(se**2)),
    }

    return resource_rows, horizon_rows, summary

# ---------------- Run only the NEW models ----------------

resource_results, horizon_results, summaries = [], [], []

for name, model in [
    ("NLinear_new", NLinearNew()),
    ("TSMixer_new", TSMixerNew()),
]:
    print("\nTraining new model:", name)
    model, best_val, seconds = train_new_model(name, model)
    rr, hh, ss = evaluate_new_model(name, model)

    resource_results.extend(rr)
    horizon_results.extend(hh)
    ss["Parameters"] = sum(p.numel() for p in model.parameters())
    ss["Best_validation_Huber"] = best_val
    ss["Training_seconds"] = seconds
    summaries.append(ss)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Persistence baseline — no training.
class PersistenceNew(nn.Module):
    def forward(self, x):
        return x[:, -1:, :].repeat(1, HORIZON, 1)

rr, hh, ss = evaluate_new_model("Persistence", PersistenceNew().to(DEVICE))
resource_results.extend(rr)
horizon_results.extend(hh)
ss["Parameters"] = 0
ss["Best_validation_Huber"] = np.nan
ss["Training_seconds"] = 0.0
summaries.append(ss)

# ---------------- Save outputs ----------------

resource_df = pd.DataFrame(resource_results)
horizon_df = pd.DataFrame(horizon_results)
summary_df = pd.DataFrame(summaries)

resource_df.to_csv(OUT / "new_models_resource_metrics.csv", index=False)
horizon_df.to_csv(OUT / "new_models_horizon_metrics.csv", index=False)
summary_df.to_csv(OUT / "new_models_summary.csv", index=False)

print("\nRESOURCE METRICS — ORIGINAL UNITS")
display(resource_df.round(4))

print("\nHORIZON-WISE METRICS")
display(horizon_df.round(4))

print("\nSUMMARY — SCALED ERRORS")
display(summary_df.sort_values("Scaled_MAE").round(5))

print("\nSaved new experiment to:", OUT)
print("New checkpoints:", CKPT)


Device: cuda
Train X: (256, 96, 4) | Train Y: (256, 12, 4)

Training new model: NLinear_new
NLinear_new | epoch 01/12 | train=0.046993 | val=0.038017
NLinear_new | epoch 02/12 | train=0.041640 | val=0.036349
NLinear_new | epoch 03/12 | train=0.040745 | val=0.035635
NLinear_new | epoch 04/12 | train=0.040390 | val=0.036100
NLinear_new | epoch 05/12 | train=0.040215 | val=0.035278
NLinear_new | epoch 06/12 | train=0.040140 | val=0.035194
NLinear_new | epoch 07/12 | train=0.040114 | val=0.035802
NLinear_new | epoch 08/12 | train=0.040092 | val=0.035417
NLinear_new | epoch 09/12 | train=0.039970 | val=0.035143
NLinear_new | epoch 10/12 | train=0.039969 | val=0.035175
NLinear_new | epoch 11/12 | train=0.039975 | val=0.035062
NLinear_new | epoch 12/12 | train=0.039963 | val=0.035081

Training new model: TSMixer_new
TSMixer_new | epoch 01/12 | train=0.119625 | val=0.095992
TSMixer_new | epoch 02/12 | train=0.083636 | val=0.070294
TSMixer_new | epoch 03/12 | train=0.068768 | val=0.062801
TSMix

,Model,Resource,MAE,RMSE,R2,WAPE_pct,sMAPE_pct
0,NLinear_new,CPU usage [%],1.733400,9.055700e+00,0.6774,35.679699,77.537903
1,NLinear_new,Memory usage [KB],245262.875000,1.300360e+06,0.5304,49.731098,72.942802
2,NLinear_new,Disk read throughput [KB/s],152.499207,2.925524e+03,0.6716,42.435200,194.828293
3,NLinear_new,Disk write throughput [KB/s],46.919701,8.244790e+02,0.4002,63.800400,136.172897
4,TSMixer_new,CPU usage [%],3.138400,9.884800e+00,0.6157,64.599503,166.723602
5,TSMixer_new,Memory usage [KB],292751.812500,1.370958e+06,0.4780,59.360199,95.526497
6,TSMixer_new,Disk read throughput [KB/s],404.901306,5.092429e+03,0.0050,112.669998,197.371902
7,TSMixer_new,Disk write throughput [KB/s],81.439499,1.065280e+03,-0.0014,110.739799,137.479004
8,Persistence,CPU usage [%],1.575000,9.476000e+00,0.6468,32.419399,25.142900
9,Persistence,Memory usage [KB],254497.734375,1.418248e+06,0.4414,51.603600,45.043098



HORIZON-WISE METRICS


,Model,Resource,Horizon_minutes,MAE,RMSE,R2
0,NLinear_new,CPU usage [%],5,0.742200,4.160500e+00,0.9324
1,NLinear_new,CPU usage [%],15,1.362200,7.217800e+00,0.7967
2,NLinear_new,CPU usage [%],30,1.778200,9.144900e+00,0.6721
3,NLinear_new,CPU usage [%],60,2.282500,1.112990e+01,0.5075
4,NLinear_new,Memory usage [KB],5,144363.484375,7.385216e+05,0.8433
5,NLinear_new,Memory usage [KB],15,216832.890625,1.159435e+06,0.6232
6,NLinear_new,Memory usage [KB],30,249385.906250,1.283865e+06,0.5459
7,NLinear_new,Memory usage [KB],60,289602.437500,1.473100e+06,0.3757
8,NLinear_new,Disk read throughput [KB/s],5,102.508003,2.922051e+03,0.7497
9,NLinear_new,Disk read throughput [KB/s],15,132.152405,2.408780e+03,0.7838



SUMMARY — SCALED ERRORS


,Model,Test_windows,Scaled_MAE,Scaled_RMSE,Parameters,Best_validation_Huber,Training_seconds
2,Persistence,100000,0.06862,0.69138,0,NaN,0.00000
0,NLinear_new,100000,0.06984,0.61698,1164,0.03506,64.79203
1,TSMixer_new,100000,0.11116,0.80916,22140,0.05947,109.36214



Saved new experiment to: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/new_forecasting_models
New checkpoints: /content/drive/MyDrive/Bitbrains_Fresh_Research/models/new_forecasting_models


In [ ]:

# ============================================================
# STEP 2: LOGISTIC REGRESSION — CPU OVERLOAD PREDICTION
# Uses existing train_loader, val_loader, test_loader, scaler
# Does NOT retrain forecasting models
# ============================================================

import os, time, joblib
import numpy as np
import pandas as pd
import torch

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler as SklearnScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

SEED = 42
np.random.seed(SEED)

OUT_DIR = (
    "/content/drive/MyDrive/Bitbrains_Fresh_Research/"
    "results/overload_prediction"
)
MODEL_DIR = (
    "/content/drive/MyDrive/Bitbrains_Fresh_Research/"
    "models/overload_prediction"
)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

CPU_INDEX = 0
CPU_THRESHOLD = 80.0  # CPU usage percentage
HORIZON_MINUTES = 60

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Target: CPU usage > 80% at least once in next 60 minutes")


# ------------------------------------------------------------
# 1. Convert each window into fixed-size statistical features
#    Input shape: [batch, 96, 4]
#    Label: any actual future CPU value > 80%
# ------------------------------------------------------------
def extract_features_and_labels(loader, split_name):
    feature_batches = []
    label_batches = []

    for X, Y in loader:
        X = X.detach().cpu().numpy()
        Y = Y.detach().cpu().numpy()

        # X and Y are assumed to be in the existing scaled space.
        # Inverse-transform Y to recover original CPU percentages.
        y_shape = Y.shape
        Y_original = scaler.inverse_transform(
            Y.reshape(-1, Y.shape[-1])
        ).reshape(y_shape)

        # Historical features only; no future values enter X_features.
        last = X[:, -1, :]
        mean = X.mean(axis=1)
        std = X.std(axis=1)
        minimum = X.min(axis=1)
        maximum = X.max(axis=1)
        slope = (X[:, -1, :] - X[:, 0, :]) / (X.shape[1] - 1)

        features = np.concatenate(
            [last, mean, std, minimum, maximum, slope], axis=1
        )

        # Positive if actual CPU exceeds 80% anywhere in the next hour.
        labels = (
            Y_original[:, :, CPU_INDEX].max(axis=1) > CPU_THRESHOLD
        ).astype(np.int32)

        feature_batches.append(features.astype(np.float32))
        label_batches.append(labels)

    X_out = np.concatenate(feature_batches, axis=0)
    y_out = np.concatenate(label_batches, axis=0)

    print(
        f"{split_name}: X={X_out.shape}, "
        f"positive={y_out.mean():.2%}, "
        f"negative={(1-y_out.mean()):.2%}"
    )
    return X_out, y_out


X_train_lr, y_train_lr = extract_features_and_labels(
    train_loader, "TRAIN"
)
X_val_lr, y_val_lr = extract_features_and_labels(
    val_loader, "VALIDATION"
)
X_test_lr, y_test_lr = extract_features_and_labels(
    test_loader, "TEST"
)


# ------------------------------------------------------------
# 2. Train Logistic Regression on training data only
# ------------------------------------------------------------
if len(np.unique(y_train_lr)) < 2:
    raise ValueError(
        "Training labels contain only one class. "
        "Check the CPU threshold and label distribution."
    )

lr_model = make_pipeline(
    SklearnScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=SEED,
        solver="lbfgs"
    )
)

start = time.time()
lr_model.fit(X_train_lr, y_train_lr)
training_seconds = time.time() - start

joblib.dump(
    lr_model, os.path.join(MODEL_DIR, "logistic_regression_cpu.joblib")
)
print(f"\nTraining completed in {training_seconds:.2f} seconds")


# ------------------------------------------------------------
# 3. Select classification threshold using validation data
#    Choose threshold maximizing validation F1.
# ------------------------------------------------------------
val_prob = lr_model.predict_proba(X_val_lr)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.05)
valid_threshold_results = []

for threshold in thresholds:
    pred = (val_prob >= threshold).astype(np.int32)
    valid_threshold_results.append(
        (
            f1_score(y_val_lr, pred, zero_division=0),
            threshold
        )
    )

best_val_f1, best_threshold = max(valid_threshold_results)

print(f"Best validation threshold: {best_threshold:.2f}")
print(f"Best validation F1: {best_val_f1:.4f}")


# ------------------------------------------------------------
# 4. Evaluate once on held-out test data
# ------------------------------------------------------------
test_prob = lr_model.predict_proba(X_test_lr)[:, 1]
test_pred = (test_prob >= best_threshold).astype(np.int32)

test_metrics = {
    "Model": "LogisticRegression_CPU_Overload",
    "CPU_threshold_pct": CPU_THRESHOLD,
    "Forecast_horizon_minutes": HORIZON_MINUTES,
    "Decision_threshold": best_threshold,
    "Test_samples": len(y_test_lr),
    "Accuracy": accuracy_score(y_test_lr, test_pred),
    "Precision": precision_score(
        y_test_lr, test_pred, zero_division=0
    ),
    "Recall": recall_score(
        y_test_lr, test_pred, zero_division=0
    ),
    "F1": f1_score(y_test_lr, test_pred, zero_division=0),
    "ROC_AUC": (
        roc_auc_score(y_test_lr, test_prob)
        if len(np.unique(y_test_lr)) == 2 else np.nan
    ),
    "Training_seconds": training_seconds
}

metrics_df = pd.DataFrame([test_metrics])
metrics_df.to_csv(
    os.path.join(OUT_DIR, "logistic_regression_test_metrics.csv"),
    index=False
)

predictions_df = pd.DataFrame({
    "Actual_CPU_Overload": y_test_lr,
    "Predicted_CPU_Overload": test_pred,
    "Overload_Probability": test_prob,
    "Decision_Threshold": best_threshold
})
predictions_df.to_csv(
    os.path.join(OUT_DIR, "logistic_regression_test_predictions.csv"),
    index=False
)

print("\nTEST METRICS")
display(metrics_df.T.rename(columns={0: "Value"}))

print("\nCONFUSION MATRIX [ [TN, FP], [FN, TP] ]")
print(confusion_matrix(y_test_lr, test_pred))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_test_lr, test_pred,
    target_names=["No CPU overload", "CPU overload"],
    zero_division=0
))

print("\nSaved model:", os.path.join(
    MODEL_DIR, "logistic_regression_cpu.joblib"
))
print("Saved outputs:", OUT_DIR)


Device: cuda
Target: CPU usage > 80% at least once in next 60 minutes
TRAIN: X=(300000, 24), positive=7.01%, negative=92.99%
VALIDATION: X=(50000, 24), positive=5.93%, negative=94.07%
TEST: X=(100000, 24), positive=3.43%, negative=96.58%

Training completed in 2.44 seconds
Best validation threshold: 0.90
Best validation F1: 0.8953

TEST METRICS


,Value
Model,LogisticRegression_CPU_Overload
CPU_threshold_pct,80.0
Forecast_horizon_minutes,60
Decision_threshold,0.9
Test_samples,100000
Accuracy,0.98617
Precision,0.82893
Recall,0.751241
F1,0.788176
ROC_AUC,0.976044



CONFUSION MATRIX [ [TN, FP], [FN, TP] ]
[[96044   531]
 [  852  2573]]

CLASSIFICATION REPORT
                 precision    recall  f1-score   support

No CPU overload       0.99      0.99      0.99     96575
   CPU overload       0.83      0.75      0.79      3425

       accuracy                           0.99    100000
      macro avg       0.91      0.87      0.89    100000
   weighted avg       0.99      0.99      0.99    100000


Saved model: /content/drive/MyDrive/Bitbrains_Fresh_Research/models/overload_prediction/logistic_regression_cpu.joblib
Saved outputs: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/overload_prediction


In [ ]:

# ============================================================
# STEP 3: OVERLOAD DECISION LAYER
# Reuse saved Logistic Regression test predictions
# ============================================================

import os
import pandas as pd
import numpy as np

BASE_DIR = (
    "/content/drive/MyDrive/Bitbrains_Fresh_Research"
)

INPUT_FILE = os.path.join(
    BASE_DIR,
    "results/overload_prediction",
    "logistic_regression_test_predictions.csv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR, "results/migration_decisions"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

THRESHOLD = 0.90

# Load existing predictions; no model retraining
pred_df = pd.read_csv(INPUT_FILE)

required = {
    "Actual_CPU_Overload",
    "Overload_Probability"
}
missing = required - set(pred_df.columns)

if missing:
    raise ValueError(f"Missing columns in predictions: {missing}")

# Convert probability into an overload decision
pred_df["Predicted_CPU_Overload"] = (
    pred_df["Overload_Probability"] >= THRESHOLD
).astype(int)

pred_df["Decision"] = np.where(
    pred_df["Predicted_CPU_Overload"] == 1,
    "OVERLOAD_ALERT",
    "NO_OVERLOAD_ALERT"
)

# Save separately; preserve original predictions file
decision_file = os.path.join(
    OUTPUT_DIR, "cpu_overload_decisions.csv"
)
pred_df.to_csv(decision_file, index=False)

# Summary
alerts = int(pred_df["Predicted_CPU_Overload"].sum())
total = len(pred_df)

print("CPU OVERLOAD DECISION SUMMARY")
print("--------------------------------")
print("Total evaluated windows:", total)
print("Overload alerts:", alerts)
print("No-overload decisions:", total - alerts)
print("Alert rate:", f"{100 * alerts / total:.2f}%")
print("Decision threshold:", THRESHOLD)
print("\nSaved:", decision_file)

display(pred_df.head(10))


CPU OVERLOAD DECISION SUMMARY
--------------------------------
Total evaluated windows: 100000
Overload alerts: 3104
No-overload decisions: 96896
Alert rate: 3.10%
Decision threshold: 0.9

Saved: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/migration_decisions/cpu_overload_decisions.csv


,Actual_CPU_Overload,Predicted_CPU_Overload,Overload_Probability,Decision_Threshold,Decision
0,0,0,0.018556,0.9,NO_OVERLOAD_ALERT
1,0,0,0.018556,0.9,NO_OVERLOAD_ALERT
2,0,0,0.018585,0.9,NO_OVERLOAD_ALERT
3,0,0,0.018562,0.9,NO_OVERLOAD_ALERT
4,0,0,0.018705,0.9,NO_OVERLOAD_ALERT
5,0,0,0.018564,0.9,NO_OVERLOAD_ALERT
6,0,0,0.018498,0.9,NO_OVERLOAD_ALERT
7,0,0,0.018531,0.9,NO_OVERLOAD_ALERT
8,0,0,0.018429,0.9,NO_OVERLOAD_ALERT
9,0,0,0.017633,0.9,NO_OVERLOAD_ALERT


In [ ]:

# ============================================================
# RECOVER CLASSIFIER DEPENDENCIES — NO FORECASTER RETRAINING
# Rebuild loaders + load saved PatchTST checkpoint
# ============================================================

import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from torch.utils.data import Dataset, DataLoader

ROOT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
SPLITS = ROOT / "data" / "splits"
SCALER_FILE = ROOT / "data" / "model_data" / "train_only_scaler.pkl"
CHECKPOINT = ROOT / "models" / "forecast_comparison" / "PatchTST_best.pt"

FEATURES = [
    "CPU usage [%]",
    "Memory usage [KB]",
    "Disk read throughput [KB/s]",
    "Disk write throughput [KB/s]",
]
LOOKBACK, HORIZON = 96, 12
BATCH_SIZE = 256
MAX_TRAIN_WINDOWS, MAX_VAL_WINDOWS, MAX_TEST_WINDOWS = 300_000, 50_000, 100_000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for p in [SCALER_FILE, CHECKPOINT]:
    if not p.exists():
        raise FileNotFoundError(
            f"Required saved artifact not found: {p}\n"
            "Check the Drive path; do not retrain automatically."
        )

with open(SCALER_FILE, "rb") as f:
    scaler = pickle.load(f)


# Same chronological split files; no cross-VM windows or time-gap windows.
class TraceWindows(Dataset):
    def __init__(self, split, max_windows, seed):
        self.arrays, self.starts = [], []
        folder = SPLITS / split

        if not folder.exists():
            raise FileNotFoundError(f"Missing split folder: {folder}")

        files = sorted(
            folder.glob("*.csv"),
            key=lambda p: int(p.stem) if p.stem.isdigit() else p.stem
        )

        for path in files:
            df = pd.read_csv(path, parse_dates=["Timestamp"])
            if not set(FEATURES).issubset(df.columns):
                continue

            ts = pd.to_datetime(df["Timestamp"], errors="coerce")
            raw = df[FEATURES].apply(pd.to_numeric, errors="coerce").to_numpy()
            values = scaler.transform(raw).astype(np.float32)

            valid = (
                ts.notna().to_numpy()
                & np.isfinite(values).all(axis=1)
            )
            valid_idx = np.flatnonzero(valid)

            if len(valid_idx) < LOOKBACK + HORIZON:
                continue

            # Divide into continuous, regular 5-minute segments.
            breaks = np.where(
                (np.diff(valid_idx) != 1)
                | (np.diff(ts.iloc[valid_idx].astype("int64").to_numpy())
                   != 300 * 10**9)
            )[0] + 1

            for segment in np.split(valid_idx, breaks):
                if len(segment) < LOOKBACK + HORIZON:
                    continue

                arr_id = len(self.arrays)
                self.arrays.append(values[segment])
                n = len(segment) - LOOKBACK - HORIZON + 1
                self.starts.extend((arr_id, s) for s in range(n))

        if not self.starts:
            raise RuntimeError(f"No valid windows found for split={split}")

        # Deterministic capped sample.
        if max_windows and len(self.starts) > max_windows:
            rng = np.random.default_rng(seed)
            ids = np.sort(rng.choice(len(self.starts), max_windows, replace=False))
            self.starts = [self.starts[i] for i in ids]

        print(f"{split}: {len(self.starts):,} windows")

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        arr_id, start = self.starts[idx]
        arr = self.arrays[arr_id]
        return (
            torch.from_numpy(arr[start:start + LOOKBACK]),
            torch.from_numpy(arr[start + LOOKBACK:start + LOOKBACK + HORIZON])
        )


train_ds = TraceWindows("train", MAX_TRAIN_WINDOWS, 42)
val_ds   = TraceWindows("val", MAX_VAL_WINDOWS, 43)
test_ds  = TraceWindows("test", MAX_TEST_WINDOWS, 44)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)


# Architecture matching the saved PatchTST checkpoint.
class PatchTST(nn.Module):
    def __init__(
        self, seq_len=96, pred_len=12, channels=4,
        patch_len=16, stride=8, d_model=128,
        n_heads=8, layers=3, ff_dim=256, dropout=0.1,
        revin=False
    ):
        super().__init__()
        self.channels = channels
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.patch_len = patch_len
        self.stride = stride
        self.revin = revin

        n_patches = (seq_len - patch_len) // stride + 1
        self.patch_proj = nn.Linear(patch_len, d_model)
        self.pos = nn.Parameter(torch.zeros(1, n_patches, d_model))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True, norm_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.head = nn.Linear(n_patches * d_model, pred_len)
        nn.init.trunc_normal_(self.pos, std=0.02)

    def forward(self, x):
        b, length, channels = x.shape
        z = x.transpose(1, 2).reshape(b * channels, length)

        if self.revin:
            mu = z.mean(dim=1, keepdim=True).detach()
            sd = torch.sqrt(
                z.var(dim=1, keepdim=True, unbiased=False) + 1e-5
            ).detach()
            z = (z - mu) / sd

        z = z.unfold(1, self.patch_len, self.stride)
        z = self.patch_proj(z)
        z = self.encoder(z + self.pos[:, :z.size(1), :])
        z = self.head(z.flatten(1))

        if self.revin:
            z = z * sd + mu

        return z.reshape(b, channels, self.pred_len).transpose(1, 2)


forecast_model = PatchTST(revin=False).to(DEVICE)
state = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True)
forecast_model.load_state_dict(state)
forecast_model.eval()

# Match the names expected by the comparison cell.
models = {"PatchTST": forecast_model}

print("\nRecovered dependencies successfully.")
print("Device:", DEVICE)
print("Loaded saved checkpoint:", CHECKPOINT)
print("PatchTST retrained: NO")
print("Loader batch shape:", tuple(next(iter(train_loader))[0].shape))


train: 300,000 windows
val: 50,000 windows
test: 100,000 windows


/tmp/ipykernel_1008/346062594.py:153: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)



Recovered dependencies successfully.
Device: cuda
Loaded saved checkpoint: /content/drive/MyDrive/Bitbrains_Fresh_Research/models/forecast_comparison/PatchTST_best.pt
PatchTST retrained: NO
Loader batch shape: (256, 96, 4)


In [ ]:

# ============================================================
# PATCHTST + CLASSIFIER COMPARISON
# No PatchTST retraining
# Logistic Regression | Random Forest | LightGBM | XGBoost
# ============================================================

import os
import json
import warnings
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

warnings.filterwarnings("ignore")

# -------------------- CONFIG --------------------
PROJECT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
RESULTS = PROJECT / "results" / "patchtst_classifier_comparison"
MODELS_DIR = PROJECT / "models" / "patchtst_classifier_comparison"

RESULTS.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

LOOKBACK = 96
HORIZON = 12
CPU_OVERLOAD_THRESHOLD = 80.0
BATCH_SIZE = 256

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -------------------- DEPENDENCY CHECK --------------------
required = ["forecast_model", "train_loader", "val_loader",
            "test_loader", "scaler"]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. Run the dependency-recovery cell first."
    )

forecast_model = forecast_model.to(DEVICE)
forecast_model.eval()

FEATURE_NAMES = [
    "CPU usage [%]",
    "Memory usage [KB]",
    "Disk read throughput [KB/s]",
    "Disk write throughput [KB/s]"
]
N_FEATURES = len(FEATURE_NAMES)

# -------------------- FEATURE ENGINEERING --------------------
# Uses historical windows and PatchTST forecasts.
# Label: maximum actual CPU in the next 12 steps exceeds 80%.
# 12 steps * 5 minutes = next 60 minutes.

def extract_features(loader, split_name):
    feature_rows = []
    labels = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            # Support loaders yielding (X, Y) or (X, Y, ...)
            X, Y = batch[0], batch[1]

            X = X.to(DEVICE, dtype=torch.float32)
            Y = Y.to(DEVICE, dtype=torch.float32)

            predictions = forecast_model(X)

            # Support models returning a tuple/list
            if isinstance(predictions, (tuple, list)):
                predictions = predictions[0]

            # Expected: [batch, horizon, features]
            if predictions.ndim != 3:
                raise ValueError(
                    f"Unexpected prediction shape: {predictions.shape}"
                )

            if predictions.shape[1] != HORIZON:
                raise ValueError(
                    f"Expected horizon={HORIZON}, got {predictions.shape[1]}"
                )

            X_np = X.detach().cpu().numpy()
            Y_np = Y.detach().cpu().numpy()
            P_np = predictions.detach().cpu().numpy()

            # Convert scaled features back to original units.
            X_raw = scaler.inverse_transform(
                X_np.reshape(-1, N_FEATURES)
            ).reshape(X_np.shape)

            Y_raw = scaler.inverse_transform(
                Y_np.reshape(-1, N_FEATURES)
            ).reshape(Y_np.shape)

            P_raw = scaler.inverse_transform(
                P_np.reshape(-1, N_FEATURES)
            ).reshape(P_np.shape)

            for i in range(len(X_raw)):
                hist = X_raw[i]
                actual_future = Y_raw[i]
                forecast = P_raw[i]

                # Historical statistics
                row = []
                for f in range(N_FEATURES):
                    series = hist[:, f]
                    row.extend([
                        np.mean(series),
                        np.std(series),
                        np.min(series),
                        np.max(series),
                        series[-1],
                        series[-1] - series[-6],
                        np.mean(series[-12:]),
                        np.mean(series[-24:])
                    ])

                # PatchTST predicted future statistics
                for f in range(N_FEATURES):
                    pred_series = forecast[:, f]
                    row.extend([
                        np.mean(pred_series),
                        np.std(pred_series),
                        np.min(pred_series),
                        np.max(pred_series),
                        pred_series[-1]
                    ])

                # Predicted CPU trend and recent CPU variability
                row.extend([
                    forecast[:, 0].max(),
                    forecast[:, 0].mean(),
                    forecast[:, 0].max() - hist[:, 0].mean(),
                    np.std(hist[-12:, 0]),
                    np.mean(hist[-12:, 0])
                ])

                # Target is derived from future actual CPU, not predictions.
                label = int(np.max(actual_future[:, 0]) > CPU_OVERLOAD_THRESHOLD)

                feature_rows.append(row)
                labels.append(label)

            if (batch_idx + 1) % 100 == 0:
                print(
                    f"{split_name}: processed "
                    f"{(batch_idx + 1) * BATCH_SIZE:,} windows"
                )

    X_features = np.asarray(feature_rows, dtype=np.float32)
    y_labels = np.asarray(labels, dtype=np.int32)

    print(
        f"{split_name}: X={X_features.shape}, "
        f"positive rate={y_labels.mean():.4f}"
    )
    return X_features, y_labels


print("\nExtracting training features...")
X_train, y_train = extract_features(train_loader, "Train")

print("\nExtracting validation features...")
X_val, y_val = extract_features(val_loader, "Validation")

print("\nExtracting test features...")
X_test, y_test = extract_features(test_loader, "Test")

# Save feature arrays so this expensive step need not be repeated.
np.savez_compressed(
    RESULTS / "classifier_features.npz",
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test
)

if len(np.unique(y_train)) < 2:
    raise ValueError("Training labels contain only one class; check the label threshold.")

if len(np.unique(y_val)) < 2:
    raise ValueError("Validation labels contain only one class; threshold tuning is not reliable.")

# -------------------- BUILD CLASSIFIERS --------------------
classifiers = {
    "LogisticRegression": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(
            max_iter=1500,
            class_weight="balanced",
            random_state=42
        )
    ),

    "RandomForest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=300,
            max_depth=18,
            min_samples_leaf=3,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        )
    )
}

try:
    from lightgbm import LGBMClassifier
    classifiers["LightGBM"] = make_pipeline(
        SimpleImputer(strategy="median"),
        LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            verbosity=-1,
            random_state=42,
            n_jobs=-1
        )
    )
    print("LightGBM available.")
except ImportError:
    print("LightGBM not installed; skipping.")

try:
    from xgboost import XGBClassifier
    classifiers["XGBoost"] = make_pipeline(
        SimpleImputer(strategy="median"),
        XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )
    )
    print("XGBoost available.")
except ImportError:
    print("XGBoost not installed; skipping.")

# -------------------- THRESHOLD TUNING --------------------
def select_threshold(y_true, probabilities):
    # Choose threshold maximizing validation F1.
    thresholds = np.arange(0.10, 0.91, 0.02)
    scores = [
        f1_score(
            y_true,
            (probabilities >= threshold).astype(int),
            zero_division=0
        )
        for threshold in thresholds
    ]
    best_index = int(np.argmax(scores))
    return float(thresholds[best_index]), float(scores[best_index])


def calculate_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)

    result = {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "positive_rate_actual": float(np.mean(y_true)),
        "positive_rate_predicted": float(np.mean(predictions)),
        "confusion_matrix": confusion_matrix(
            y_true, predictions, labels=[0, 1]
        ).tolist()
    }

    if len(np.unique(y_true)) == 2:
        result["roc_auc"] = roc_auc_score(y_true, probabilities)
        result["average_precision"] = average_precision_score(
            y_true, probabilities
        )
    else:
        result["roc_auc"] = float("nan")
        result["average_precision"] = float("nan")

    return result


# -------------------- TRAIN / VALIDATE / TEST --------------------
all_results = []
fitted_models = {}

for name, clf in classifiers.items():
    print(f"\n{'=' * 60}\nTraining {name}\n{'=' * 60}")

    clf.fit(X_train, y_train)

    val_prob = clf.predict_proba(X_val)[:, 1]
    threshold, val_best_f1 = select_threshold(y_val, val_prob)

    test_prob = clf.predict_proba(X_test)[:, 1]
    metrics = calculate_metrics(y_test, test_prob, threshold)

    metrics["classifier"] = name
    metrics["validation_best_f1"] = val_best_f1
    all_results.append(metrics)
    fitted_models[name] = clf

    # Save the fitted pipeline
    import joblib
    joblib.dump(clf, MODELS_DIR / f"{name}.joblib")

    print(f"Validation best F1: {val_best_f1:.4f}")
    print(f"Test accuracy:      {metrics['accuracy']:.4f}")
    print(f"Test precision:     {metrics['precision']:.4f}")
    print(f"Test recall:        {metrics['recall']:.4f}")
    print(f"Test F1:            {metrics['f1']:.4f}")
    print(f"Test ROC-AUC:       {metrics['roc_auc']:.4f}")
    print(f"Test Avg Precision: {metrics['average_precision']:.4f}")
    print("Test confusion matrix [[TN, FP], [FN, TP]]:")
    print(metrics["confusion_matrix"])

# -------------------- RANK AND SAVE RESULTS --------------------
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(
    by=["f1", "recall", "roc_auc"],
    ascending=False
).reset_index(drop=True)

results_df.to_csv(RESULTS / "classifier_comparison.csv", index=False)

with open(RESULTS / "classifier_comparison.json", "w") as f:
    json.dump(all_results, f, indent=2, allow_nan=True)

print("\n\n========== FINAL CLASSIFIER COMPARISON ==========")
display_cols = [
    "classifier", "threshold", "accuracy", "precision",
    "recall", "f1", "roc_auc", "average_precision"
]
print(results_df[display_cols].to_string(index=False))

print("\nSaved results to:", RESULTS)
print("Saved fitted classifiers to:", MODELS_DIR)
print("\nPatchTST was used for inference only; it was NOT retrained.")


Device: cuda

Extracting training features...
Train: processed 25,600 windows
Train: processed 51,200 windows
Train: processed 76,800 windows
Train: processed 102,400 windows
Train: processed 128,000 windows
Train: processed 153,600 windows
Train: processed 179,200 windows
Train: processed 204,800 windows
Train: processed 230,400 windows
Train: processed 256,000 windows
Train: processed 281,600 windows
Train: X=(300000, 57), positive rate=0.0701

Extracting validation features...
Validation: processed 25,600 windows
Validation: X=(50000, 57), positive rate=0.0593

Extracting test features...
Test: processed 25,600 windows
Test: processed 51,200 windows
Test: processed 76,800 windows
Test: X=(100000, 57), positive rate=0.0343
LightGBM available.
XGBoost available.

Training LogisticRegression
Validation best F1: 0.9217
Test accuracy:      0.9878
Test precision:     0.8346
Test recall:        0.8029
Test F1:            0.8185
Test ROC-AUC:       0.9768
Test Avg Precision: 0.8707
Test con

In [ ]:

# ============================================================
# CLASSIFIER RESULTS AUDIT — NO MODEL RETRAINING
# ============================================================

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix
)

ROOT = Path(
    "/content/drive/MyDrive/Bitbrains_Fresh_Research"
)
RESULTS = ROOT / "results" / "patchtst_classifier_comparison"
MODELS = ROOT / "models" / "patchtst_classifier_comparison"

# 1. Load the saved feature arrays
data = np.load(RESULTS / "classifier_features.npz")

X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]

print("=" * 65)
print("1. LABEL DISTRIBUTION")
print("=" * 65)

for split, y in [
    ("Validation", y_val),
    ("Test", y_test)
]:
    positives = int(y.sum())
    negatives = len(y) - positives
    print(
        f"{split}: total={len(y):,}, "
        f"overload={positives:,} ({y.mean():.3%}), "
        f"no-overload={negatives:,} ({1-y.mean():.3%})"
    )

# 2. Majority-class baseline
baseline_pred = np.zeros_like(y_test)
print("\n" + "=" * 65)
print("2. MAJORITY-CLASS BASELINE")
print("=" * 65)
print(f"Accuracy:  {accuracy_score(y_test, baseline_pred):.4f}")
print(f"Precision: {precision_score(y_test, baseline_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, baseline_pred, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_test, baseline_pred, zero_division=0):.4f}")

# 3. Re-evaluate saved classifiers using validation-only threshold tuning.
# The test labels are NOT used to choose the threshold.
print("\n" + "=" * 65)
print("3. VALIDATION-TUNED THRESHOLDS AND TEST METRICS")
print("=" * 65)

model_files = {
    "LogisticRegression": MODELS / "LogisticRegression.joblib",
    "RandomForest": MODELS / "RandomForest.joblib",
    "LightGBM": MODELS / "LightGBM.joblib",
    "XGBoost": MODELS / "XGBoost.joblib",
}

audit_rows = []

# Fine threshold grid; validation data only selects the threshold.
threshold_grid = np.arange(0.01, 1.00, 0.01)

for name, path in model_files.items():
    if not path.exists():
        print(f"\nSkipping {name}: saved model not found.")
        continue

    model = joblib.load(path)

    val_prob = model.predict_proba(X_val)[:, 1]
    test_prob = model.predict_proba(X_test)[:, 1]

    val_f1 = [
        f1_score(
            y_val,
            (val_prob >= t).astype(int),
            zero_division=0
        )
        for t in threshold_grid
    ]

    best_idx = int(np.argmax(val_f1))
    best_threshold = float(threshold_grid[best_idx])

    test_pred = (test_prob >= best_threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_test, test_pred, labels=[0, 1]
    ).ravel()

    row = {
        "classifier": name,
        "threshold": best_threshold,
        "validation_F1": val_f1[best_idx],
        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_precision": precision_score(
            y_test, test_pred, zero_division=0
        ),
        "test_recall": recall_score(
            y_test, test_pred, zero_division=0
        ),
        "test_F1": f1_score(
            y_test, test_pred, zero_division=0
        ),
        "test_ROC_AUC": roc_auc_score(y_test, test_prob),
        "test_average_precision": average_precision_score(
            y_test, test_prob
        ),
        "TN": int(tn), "FP": int(fp),
        "FN": int(fn), "TP": int(tp),
        "threshold_at_grid_boundary": (
            best_idx == 0 or best_idx == len(threshold_grid) - 1
        )
    }
    audit_rows.append(row)

    print(f"\n{name}")
    print(f"  Selected threshold: {best_threshold:.2f}")
    print(f"  Validation F1:      {val_f1[best_idx]:.4f}")
    print(f"  Test accuracy:      {row['test_accuracy']:.4f}")
    print(f"  Test precision:     {row['test_precision']:.4f}")
    print(f"  Test recall:        {row['test_recall']:.4f}")
    print(f"  Test F1:             {row['test_F1']:.4f}")
    print(f"  Test ROC-AUC:        {row['test_ROC_AUC']:.4f}")
    print(f"  Test average precision: {row['test_average_precision']:.4f}")
    print(f"  TN={tn:,}, FP={fp:,}, FN={fn:,}, TP={tp:,}")
    print(f"  Best threshold at grid boundary: {row['threshold_at_grid_boundary']}")

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(RESULTS / "classifier_audit.csv", index=False)

print("\nAudit saved to:", RESULTS / "classifier_audit.csv")
display(audit_df)


1. LABEL DISTRIBUTION
Validation: total=50,000, overload=2,965 (5.930%), no-overload=47,035 (94.070%)
Test: total=100,000, overload=3,425 (3.425%), no-overload=96,575 (96.575%)

2. MAJORITY-CLASS BASELINE
Accuracy:  0.9657
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000

3. VALIDATION-TUNED THRESHOLDS AND TEST METRICS

LogisticRegression
  Selected threshold: 0.93
  Validation F1:      0.9249
  Test accuracy:      0.9880
  Test precision:     0.8593
  Test recall:        0.7755
  Test F1:             0.8152
  Test ROC-AUC:        0.9768
  Test average precision: 0.8707
  TN=96,140, FP=435, FN=769, TP=2,656
  Best threshold at grid boundary: False

RandomForest
  Selected threshold: 0.70
  Validation F1:      0.9276
  Test accuracy:      0.9881
  Test precision:     0.8577
  Test recall:        0.7813
  Test F1:             0.8177
  Test ROC-AUC:        0.9945
  Test average precision: 0.9120
  TN=96,131, FP=444, FN=749, TP=2,676
  Best threshold at grid boundary: False


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



LightGBM
  Selected threshold: 0.90
  Validation F1:      0.9302
  Test accuracy:      0.9882
  Test precision:     0.8911
  Test recall:        0.7454
  Test F1:             0.8118
  Test ROC-AUC:        0.9942
  Test average precision: 0.9085
  TN=96,263, FP=312, FN=872, TP=2,553
  Best threshold at grid boundary: False

XGBoost
  Selected threshold: 0.47
  Validation F1:      0.9304
  Test accuracy:      0.9882
  Test precision:     0.8737
  Test recall:        0.7655
  Test F1:             0.8161
  Test ROC-AUC:        0.9944
  Test average precision: 0.9130
  TN=96,196, FP=379, FN=803, TP=2,622
  Best threshold at grid boundary: False

Audit saved to: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/patchtst_classifier_comparison/classifier_audit.csv


,classifier,threshold,validation_F1,test_accuracy,test_precision,test_recall,test_F1,test_ROC_AUC,test_average_precision,TN,FP,FN,TP,threshold_at_grid_boundary
0,LogisticRegression,0.93,0.924892,0.98796,0.859269,0.775474,0.815224,0.976753,0.870707,96140,435,769,2656,False
1,RandomForest,0.70,0.927617,0.98807,0.857692,0.781314,0.817723,0.994456,0.911997,96131,444,749,2676,False
2,LightGBM,0.90,0.930159,0.98816,0.891099,0.745401,0.811765,0.994154,0.908466,96263,312,872,2553,False
3,XGBoost,0.47,0.930386,0.98818,0.873709,0.765547,0.816060,0.994398,0.913011,96196,379,803,2622,False


In [ ]:

# ============================================================
# GENERALIZATION AUDIT — NO RETRAINING
# Checks train/validation/test performance at the same threshold
# ============================================================

import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)

ROOT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
RESULTS = ROOT / "results" / "patchtst_classifier_comparison"
MODEL_DIR = ROOT / "models" / "patchtst_classifier_comparison"

data = np.load(RESULTS / "classifier_features.npz")

splits = {
    "Train": (data["X_train"], data["y_train"]),
    "Validation": (data["X_val"], data["y_val"]),
    "Test": (data["X_test"], data["y_test"]),
}

model_names = [
    "LogisticRegression",
    "RandomForest",
    "LightGBM",
    "XGBoost",
]

rows = []

for name in model_names:
    path = MODEL_DIR / f"{name}.joblib"
    if not path.exists():
        print(f"Skipping missing model: {name}")
        continue

    model = joblib.load(path)

    # Select threshold using validation data ONLY.
    Xv, yv = splits["Validation"]
    p_val = model.predict_proba(Xv)[:, 1]

    thresholds = np.arange(0.01, 1.00, 0.01)
    val_f1s = [
        f1_score(yv, (p_val >= t).astype(int), zero_division=0)
        for t in thresholds
    ]
    threshold = float(thresholds[int(np.argmax(val_f1s))])

    print(f"\n{'=' * 60}\n{name} | threshold={threshold:.2f}\n")

    for split_name, (X, y) in splits.items():
        p = model.predict_proba(X)[:, 1]
        pred = (p >= threshold).astype(int)

        row = {
            "classifier": name,
            "split": split_name,
            "positive_rate": float(y.mean()),
            "predicted_positive_rate": float(pred.mean()),
            "accuracy": accuracy_score(y, pred),
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0),
            "roc_auc": (
                roc_auc_score(y, p)
                if len(np.unique(y)) == 2 else np.nan
            ),
            "average_precision": average_precision_score(y, p),
            "threshold": threshold,
        }
        rows.append(row)

        print(
            f"{split_name:10s} | "
            f"prevalence={row['positive_rate']:.3%} | "
            f"accuracy={row['accuracy']:.4f} | "
            f"precision={row['precision']:.4f} | "
            f"recall={row['recall']:.4f} | "
            f"F1={row['f1']:.4f} | "
            f"AUC={row['roc_auc']:.4f} | "
            f"AP={row['average_precision']:.4f}"
        )

audit = pd.DataFrame(rows)
audit.to_csv(RESULTS / "generalization_audit.csv", index=False)

print("\nSaved:", RESULTS / "generalization_audit.csv")
display(audit)



LogisticRegression | threshold=0.93

Train      | prevalence=7.012% | accuracy=0.9864 | precision=0.9222 | recall=0.8801 | F1=0.9007 | AUC=0.9873 | AP=0.9399
Validation | prevalence=5.930% | accuracy=0.9913 | precision=0.9521 | recall=0.8992 | F1=0.9249 | AUC=0.9894 | AP=0.9496
Test       | prevalence=3.425% | accuracy=0.9880 | precision=0.8593 | recall=0.7755 | F1=0.8152 | AUC=0.9768 | AP=0.8707

RandomForest | threshold=0.70

Train      | prevalence=7.012% | accuracy=0.9934 | precision=0.9535 | recall=0.9525 | F1=0.9530 | AUC=0.9995 | AP=0.9944
Validation | prevalence=5.930% | accuracy=0.9917 | precision=0.9557 | recall=0.9012 | F1=0.9276 | AUC=0.9976 | AP=0.9790
Test       | prevalence=3.425% | accuracy=0.9881 | precision=0.8577 | recall=0.7813 | F1=0.8177 | AUC=0.9945 | AP=0.9120


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



LightGBM | threshold=0.90



/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train      | prevalence=7.012% | accuracy=0.9912 | precision=0.9615 | recall=0.9103 | F1=0.9352 | AUC=0.9990 | AP=0.9878
Validation | prevalence=5.930% | accuracy=0.9920 | precision=0.9669 | recall=0.8961 | F1=0.9302 | AUC=0.9977 | AP=0.9788


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Test       | prevalence=3.425% | accuracy=0.9882 | precision=0.8911 | recall=0.7454 | F1=0.8118 | AUC=0.9942 | AP=0.9085

XGBoost | threshold=0.47

Train      | prevalence=7.012% | accuracy=0.9921 | precision=0.9658 | recall=0.9200 | F1=0.9423 | AUC=0.9987 | AP=0.9877
Validation | prevalence=5.930% | accuracy=0.9920 | precision=0.9612 | recall=0.9015 | F1=0.9304 | AUC=0.9978 | AP=0.9792
Test       | prevalence=3.425% | accuracy=0.9882 | precision=0.8737 | recall=0.7655 | F1=0.8161 | AUC=0.9944 | AP=0.9130

Saved: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/patchtst_classifier_comparison/generalization_audit.csv


,classifier,split,positive_rate,predicted_positive_rate,accuracy,precision,recall,f1,roc_auc,average_precision,threshold
0,LogisticRegression,Train,0.07012,0.066917,0.986390,0.922242,0.880110,0.900684,0.987334,0.939913,0.93
1,LogisticRegression,Validation,0.05930,0.056000,0.991340,0.952143,0.899157,0.924892,0.989394,0.949565,0.93
2,LogisticRegression,Test,0.03425,0.030910,0.987960,0.859269,0.775474,0.815224,0.976753,0.870707,0.93
3,RandomForest,Train,0.07012,0.070050,0.993410,0.953462,0.952510,0.952986,0.999541,0.994448,0.70
4,RandomForest,Validation,0.05930,0.055920,0.991660,0.955651,0.901180,0.927617,0.997586,0.979017,0.70
5,RandomForest,Test,0.03425,0.031200,0.988070,0.857692,0.781314,0.817723,0.994456,0.911997,0.70
6,LightGBM,Train,0.07012,0.066390,0.991157,0.961490,0.910344,0.935218,0.998953,0.987812,0.90
7,LightGBM,Validation,0.05930,0.054960,0.992020,0.966885,0.896121,0.930159,0.997696,0.978788,0.90
8,LightGBM,Test,0.03425,0.028650,0.988160,0.891099,0.745401,0.811765,0.994154,0.908466,0.90
9,XGBoost,Train,0.07012,0.066793,0.992107,0.965815,0.919994,0.942348,0.998719,0.987730,0.47


In [ ]:

# ============================================================
# ABLATION: HISTORY-ONLY vs HISTORY + PatchTST FEATURES
# PatchTST is NOT retrained.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    accuracy_score, roc_auc_score, average_precision_score
)

ROOT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
RESULTS = ROOT / "results" / "patchtst_classifier_comparison"

data = np.load(RESULTS / "classifier_features.npz")

Xtr, ytr = data["X_train"], data["y_train"]
Xv, yv = data["X_val"], data["y_val"]
Xte, yte = data["X_test"], data["y_test"]

# Feature construction in the earlier cell:
# 4 resources * 8 historical statistics = 32 features
# 4 resources * 5 forecast statistics = 20 features
# 5 additional CPU statistics = 5 features
assert Xtr.shape[1] == 57, f"Expected 57 features, got {Xtr.shape[1]}"

feature_sets = {
    "History_only": slice(0, 32),
    "History_plus_PatchTST": slice(0, 57),
}

rows = []

for feature_set, cols in feature_sets.items():
    A, B, C = Xtr[:, cols], Xv[:, cols], Xte[:, cols]

    for model_name, model in {
        "LogisticRegression": make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            LogisticRegression(
                max_iter=1500, class_weight="balanced",
                random_state=42
            )
        ),
        "RandomForest": make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestClassifier(
                n_estimators=300, max_depth=18,
                min_samples_leaf=3, class_weight="balanced",
                n_jobs=-1, random_state=42
            )
        )
    }.items():

        model.fit(A, ytr)

        pv = model.predict_proba(B)[:, 1]
        pt = model.predict_proba(C)[:, 1]

        thresholds = np.arange(0.01, 1.00, 0.01)
        scores = [
            f1_score(yv, (pv >= t).astype(int), zero_division=0)
            for t in thresholds
        ]
        threshold = float(thresholds[int(np.argmax(scores))])
        pred = (pt >= threshold).astype(int)

        row = {
            "features": feature_set,
            "classifier": model_name,
            "threshold": threshold,
            "validation_F1": max(scores),
            "test_accuracy": accuracy_score(yte, pred),
            "test_precision": precision_score(
                yte, pred, zero_division=0
            ),
            "test_recall": recall_score(
                yte, pred, zero_division=0
            ),
            "test_F1": f1_score(
                yte, pred, zero_division=0
            ),
            "test_ROC_AUC": roc_auc_score(yte, pt),
            "test_average_precision": average_precision_score(yte, pt),
        }
        rows.append(row)

result = pd.DataFrame(rows)
result.to_csv(RESULTS / "history_vs_patchtst_ablation.csv", index=False)

print(result.to_string(index=False))
print("\nSaved:", RESULTS / "history_vs_patchtst_ablation.csv")


             features         classifier  threshold  validation_F1  test_accuracy  test_precision  test_recall  test_F1  test_ROC_AUC  test_average_precision
         History_only LogisticRegression       0.96       0.909595        0.98591        0.864953     0.697518 0.772264      0.976451                0.852261
         History_only       RandomForest       0.67       0.924463        0.98819        0.855063     0.788905 0.820653      0.994803                0.911843
History_plus_PatchTST LogisticRegression       0.93       0.924892        0.98796        0.859269     0.775474 0.815224      0.976753                0.870707
History_plus_PatchTST       RandomForest       0.70       0.927617        0.98807        0.857692     0.781314 0.817723      0.994456                0.911997

Saved: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/patchtst_classifier_comparison/history_vs_patchtst_ablation.csv


In [ ]:

# ============================================================
# CORRECTED ABLATION: FAIR HISTORY-ONLY vs PATCHTST FEATURES
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)

ROOT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
OUT = ROOT / "results" / "patchtst_classifier_comparison"
d = np.load(OUT / "classifier_features.npz")

Xtr, ytr = d["X_train"], d["y_train"]
Xv, yv = d["X_val"], d["y_val"]
Xte, yte = d["X_test"], d["y_test"]

assert Xtr.shape[1] == 57

# Features 0-31: historical statistics for all 4 resources.
# Features 32-51: PatchTST forecast statistics.
# Features 52-54: forecast-related CPU statistics.
# Features 55-56: historical CPU standard deviation and mean.
history_cols = list(range(32)) + [55, 56]

feature_sets = {
    "History_only": history_cols,
    "History_plus_PatchTST": list(range(57))
}

rows = []

for feature_name, cols in feature_sets.items():
    for model_name, estimator in {
        "LogisticRegression": make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            LogisticRegression(
                max_iter=1500,
                class_weight="balanced",
                random_state=42
            )
        ),
        "RandomForest": make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestClassifier(
                n_estimators=300,
                max_depth=18,
                min_samples_leaf=3,
                class_weight="balanced",
                n_jobs=-1,
                random_state=42
            )
        )
    }.items():

        estimator.fit(Xtr[:, cols], ytr)

        val_p = estimator.predict_proba(Xv[:, cols])[:, 1]
        test_p = estimator.predict_proba(Xte[:, cols])[:, 1]

        thresholds = np.arange(0.01, 1.00, 0.01)
        val_f1 = [
            f1_score(
                yv, (val_p >= t).astype(int), zero_division=0
            )
            for t in thresholds
        ]
        threshold = float(thresholds[np.argmax(val_f1)])
        pred = (test_p >= threshold).astype(int)

        rows.append({
            "features": feature_name,
            "classifier": model_name,
            "threshold": threshold,
            "validation_F1": max(val_f1),
            "test_accuracy": accuracy_score(yte, pred),
            "test_precision": precision_score(
                yte, pred, zero_division=0
            ),
            "test_recall": recall_score(
                yte, pred, zero_division=0
            ),
            "test_F1": f1_score(
                yte, pred, zero_division=0
            ),
            "test_ROC_AUC": roc_auc_score(yte, test_p),
            "test_average_precision": average_precision_score(
                yte, test_p
            )
        })

result = pd.DataFrame(rows)
result.to_csv(OUT / "corrected_ablation.csv", index=False)
print(result.to_string(index=False))
print("\nSaved:", OUT / "corrected_ablation.csv")


             features         classifier  threshold  validation_F1  test_accuracy  test_precision  test_recall  test_F1  test_ROC_AUC  test_average_precision
         History_only LogisticRegression       0.96       0.913921        0.98637        0.861754     0.717080 0.782789      0.976696                0.866370
         History_only       RandomForest       0.57       0.926077        0.98818        0.822178     0.835620 0.828844      0.994727                0.912168
History_plus_PatchTST LogisticRegression       0.93       0.924731        0.98794        0.859178     0.774891 0.814860      0.976749                0.870660
History_plus_PatchTST       RandomForest       0.70       0.927617        0.98807        0.857692     0.781314 0.817723      0.994456                0.911997

Saved: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/patchtst_classifier_comparison/corrected_ablation.csv


In [ ]:

# ============================================================
# ADAPTIVE MULTI-RESOURCE RISK CLASSIFIER
# Reuses PatchTST; DOES NOT retrain the forecasting model.
# Learns resource reference levels from TRAIN data only.
# Validation selects the probability threshold.
# Test is for evaluation only.
# ============================================================

import json
import time
import joblib
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

ROOT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
OUT = ROOT / "results" / "adaptive_multi_resource"
MODEL_DIR = ROOT / "models" / "adaptive_multi_resource"
OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Required variables should already exist from cell 21.
for name in ["forecast_model", "train_loader", "val_loader",
             "test_loader", "scaler"]:
    if name not in globals():
        raise RuntimeError(
            f"{name} is missing. Run the dependency-recovery cell 21 first."
        )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
forecast_model = forecast_model.to(DEVICE)
forecast_model.eval()

FEATURES = [
    "CPU usage [%]",
    "Memory usage [KB]",
    "Disk read throughput [KB/s]",
    "Disk write throughput [KB/s]"
]

N_FEATURES = len(FEATURES)
REFERENCE_QUANTILE = 0.95
SEED = 42

print("Device:", DEVICE)
print("Forecast model: loaded existing checkpoint")
print("Decision target: adaptive high demand across four resources")


# ------------------------------------------------------------
# 1. Extract historical and forecast-derived features.
#    Labels are not assigned yet because resource reference
#    levels must first be learned from training data.
# ------------------------------------------------------------
def extract_split(loader, split_name):
    all_features = []
    all_future_max = []

    with torch.no_grad():
        for batch in loader:
            X, Y = batch[0], batch[1]
            X = X.to(DEVICE, dtype=torch.float32)
            Y = Y.to(DEVICE, dtype=torch.float32)

            P = forecast_model(X)
            if isinstance(P, (tuple, list)):
                P = P[0]

            if P.ndim != 3 or P.shape[-1] != N_FEATURES:
                raise ValueError(f"Unexpected forecast shape: {P.shape}")

            Xn = X.cpu().numpy()
            Yn = Y.cpu().numpy()
            Pn = P.cpu().numpy()

            Xraw = scaler.inverse_transform(
                Xn.reshape(-1, N_FEATURES)
            ).reshape(Xn.shape)

            Yraw = scaler.inverse_transform(
                Yn.reshape(-1, N_FEATURES)
            ).reshape(Yn.shape)

            Praw = scaler.inverse_transform(
                Pn.reshape(-1, N_FEATURES)
            ).reshape(Pn.shape)

            # Historical features: 7 statistics per resource.
            hist_mean = Xraw.mean(axis=1)
            hist_std = Xraw.std(axis=1)
            hist_max = Xraw.max(axis=1)
            hist_min = Xraw.min(axis=1)
            hist_last = Xraw[:, -1, :]
            hist_recent = Xraw[:, -12:, :].mean(axis=1)
            hist_slope = (Xraw[:, -1, :] - Xraw[:, 0, :]) / 95.0

            # Forecast features: 5 statistics per resource.
            pred_mean = Praw.mean(axis=1)
            pred_std = Praw.std(axis=1)
            pred_max = Praw.max(axis=1)
            pred_min = Praw.min(axis=1)
            pred_slope = Praw[:, -1, :] - Praw[:, 0, :]

            # 28 historical + 20 forecast-derived features.
            feats = np.concatenate([
                hist_mean, hist_std, hist_max, hist_min,
                hist_last, hist_recent, hist_slope,
                pred_mean, pred_std, pred_max, pred_min, pred_slope
            ], axis=1)

            # Target candidates: maximum actual demand in the next hour.
            future_max = Yraw.max(axis=1)

            all_features.append(feats.astype(np.float32))
            all_future_max.append(future_max.astype(np.float32))

    Xout = np.concatenate(all_features)
    future_max = np.concatenate(all_future_max)

    print(f"{split_name}: features={Xout.shape}, "
          f"future resource maxima={future_max.shape}")

    return Xout, future_max


start = time.time()
X_train, train_future_max = extract_split(train_loader, "TRAIN")
X_val, val_future_max = extract_split(val_loader, "VALIDATION")
X_test, test_future_max = extract_split(test_loader, "TEST")


# ------------------------------------------------------------
# 2. Learn reference levels from TRAIN only.
#    These are statistical high-demand references, NOT host
#    capacity limits. No fixed 80% CPU rule is used.
# ------------------------------------------------------------
resource_reference = np.quantile(
    train_future_max, REFERENCE_QUANTILE, axis=0
)

reference_table = pd.DataFrame({
    "resource": FEATURES,
    "training_quantile": REFERENCE_QUANTILE,
    "learned_reference": resource_reference
})

print("\nTRAIN-LEARNED RESOURCE REFERENCES")
print(reference_table.to_string(index=False))

reference_table.to_csv(
    OUT / "learned_resource_references.csv", index=False
)


# Positive if ANY resource's future maximum exceeds its
# corresponding train-learned reference level.
def make_labels(future_max):
    return (future_max > resource_reference[None, :]).any(axis=1).astype(int)


y_train = make_labels(train_future_max)
y_val = make_labels(val_future_max)
y_test = make_labels(test_future_max)

for name, y in [
    ("Train", y_train), ("Validation", y_val), ("Test", y_test)
]:
    print(f"{name} positive rate: {y.mean():.3%} "
          f"({int(y.sum())}/{len(y)})")

if len(np.unique(y_train)) < 2 or len(np.unique(y_val)) < 2:
    raise ValueError(
        "The learned labels do not contain both classes in train/validation. "
        "Inspect the learned resource references before proceeding."
    )


# ------------------------------------------------------------
# 3. Fit Logistic Regression using TRAIN only.
# ------------------------------------------------------------
risk_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=1500,
        random_state=SEED
    )
)

risk_model.fit(X_train, y_train)
joblib.dump(risk_model, MODEL_DIR / "adaptive_logistic_regression.joblib")


# ------------------------------------------------------------
# 4. Select the probability threshold using VALIDATION only.
# ------------------------------------------------------------
val_prob = risk_model.predict_proba(X_val)[:, 1]
test_prob = risk_model.predict_proba(X_test)[:, 1]

threshold_grid = np.arange(0.05, 0.96, 0.01)
val_f1_scores = [
    f1_score(y_val, val_prob >= t, zero_division=0)
    for t in threshold_grid
]
best_idx = int(np.argmax(val_f1_scores))
risk_threshold = float(threshold_grid[best_idx])

print(f"\nValidation-selected risk threshold: {risk_threshold:.2f}")
print(f"Validation F1 at selected threshold: {val_f1_scores[best_idx]:.4f}")


# ------------------------------------------------------------
# 5. Evaluate ONCE on test data.
# ------------------------------------------------------------
test_pred = (test_prob >= risk_threshold).astype(int)

metrics = {
    "target": "Any resource exceeds its train-learned 95th-percentile reference",
    "reference_quantile": REFERENCE_QUANTILE,
    "probability_threshold": risk_threshold,
    "test_prevalence": float(y_test.mean()),
    "test_accuracy": float(accuracy_score(y_test, test_pred)),
    "test_precision": float(precision_score(y_test, test_pred, zero_division=0)),
    "test_recall": float(recall_score(y_test, test_pred, zero_division=0)),
    "test_f1": float(f1_score(y_test, test_pred, zero_division=0)),
    "test_roc_auc": float(roc_auc_score(y_test, test_prob)),
    "test_average_precision": float(average_precision_score(y_test, test_prob)),
    "confusion_matrix_TN_FP_FN_TP": confusion_matrix(
        y_test, test_pred, labels=[0, 1]
    ).tolist()
}

print("\nADAPTIVE MULTI-RESOURCE TEST RESULTS")
print(json.dumps(metrics, indent=2))

with open(OUT / "adaptive_classifier_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

pd.DataFrame({
    "actual_high_demand": y_test,
    "risk_probability": test_prob,
    "risk_alert": test_pred
}).to_csv(OUT / "adaptive_test_predictions.csv", index=False)

print("\nSaved reference levels, classifier, metrics, and test predictions.")
print("Output folder:", OUT)
print(f"Elapsed time: {(time.time() - start) / 60:.2f} minutes")


Device: cuda
Forecast model: loaded existing checkpoint
Decision target: adaptive high demand across four resources
TRAIN: features=(300000, 48), future resource maxima=(300000, 4)
VALIDATION: features=(50000, 48), future resource maxima=(50000, 4)
TEST: features=(100000, 48), future resource maxima=(100000, 4)

TRAIN-LEARNED RESOURCE REFERENCES
                    resource  training_quantile  learned_reference
               CPU usage [%]               0.95       9.905000e+01
           Memory usage [KB]               0.95       6.174014e+06
 Disk read throughput [KB/s]               0.95       1.650021e+02
Disk write throughput [KB/s]               0.95       2.877333e+02
Train positive rate: 13.655% (40964/300000)
Validation positive rate: 13.710% (6855/50000)
Test positive rate: 11.227% (11227/100000)

Validation-selected risk threshold: 0.66
Validation F1 at selected threshold: 0.8076

ADAPTIVE MULTI-RESOURCE TEST RESULTS
{
  "target": "Any resource exceeds its train-learned 95th-

In [ ]:

# ============================================================
# CLOUDSIM BRIDGE EXPORT — TIMESTAMPED VM FORECASTS + RISK
# Run after notebook cell 27.
# No model training. Raw dataset remains untouched.
# ============================================================

import numpy as np
import pandas as pd
import torch
import joblib
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Bitbrains_Fresh_Research")
TEST_DIR = ROOT / "data" / "splits" / "test"
OUT_DIR = ROOT / "results" / "cloudsim"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = [
    "CPU usage [%]",
    "Memory usage [KB]",
    "Disk read throughput [KB/s]",
    "Disk write throughput [KB/s]",
]
LOOKBACK = 96
HORIZON = 12
STEP_MINUTES = 15
EXPORT_EVERY_STEPS = STEP_MINUTES // 5

assert EXPORT_EVERY_STEPS >= 1
assert "forecast_model" in globals(), "Run recovery cell 21 first."
assert "scaler" in globals(), "Train-only scaler is missing."
assert "risk_model" in globals(), "Run adaptive classifier cell 27."
assert "risk_threshold" in globals(), "Classifier threshold is missing."

forecast_model.eval()
device = next(forecast_model.parameters()).device

records = []
files = sorted(
    TEST_DIR.glob("*.csv"),
    key=lambda p: int(p.stem) if p.stem.isdigit() else p.stem
)

if not files:
    raise FileNotFoundError(f"No test CSVs found in {TEST_DIR}")

def make_risk_features(history, prediction):
    # Shapes: history [96, 4], prediction [12, 4].
    # Preserve the exact feature ordering used to train cell 27.
    hist_mean = history.mean(axis=0)
    hist_std = history.std(axis=0)
    hist_max = history.max(axis=0)
    hist_min = history.min(axis=0)
    hist_last = history[-1]
    hist_recent = history[-12:].mean(axis=0)
    hist_slope = (history[-1] - history[0]) / 95.0

    pred_mean = prediction.mean(axis=0)
    pred_std = prediction.std(axis=0)
    pred_max = prediction.max(axis=0)
    pred_min = prediction.min(axis=0)
    pred_slope = prediction[-1] - prediction[0]

    return np.concatenate([
        hist_mean, hist_std, hist_max, hist_min,
        hist_last, hist_recent, hist_slope,
        pred_mean, pred_std, pred_max, pred_min, pred_slope
    ]).astype(np.float32)

with torch.no_grad():
    for file_idx, path in enumerate(files, start=1):
        df = pd.read_csv(path)
        if not set(["Timestamp"] + FEATURES).issubset(df.columns):
            continue

        ts = pd.to_datetime(df["Timestamp"], errors="coerce")
        raw = df[FEATURES].apply(
            pd.to_numeric, errors="coerce"
        ).to_numpy(dtype=np.float64)

        valid = ts.notna().to_numpy() & np.isfinite(raw).all(axis=1)
        valid_idx = np.flatnonzero(valid)

        if len(valid_idx) < LOOKBACK + HORIZON:
            continue

        # Break at missing rows, non-consecutive rows or timestamp gaps.
        ts_ns = ts.astype("int64").to_numpy()
        breaks = np.where(
            (np.diff(valid_idx) != 1) |
            (np.diff(ts_ns[valid_idx]) != 300 * 10**9)
        )[0] + 1

        segments = np.split(valid_idx, breaks)
        vm_id = int(path.stem) if path.stem.isdigit() else path.stem

        for segment in segments:
            if len(segment) < LOOKBACK + HORIZON:
                continue

            values_raw = raw[segment]
            values_scaled = scaler.transform(values_raw).astype(np.float32)
            segment_ts = ts.iloc[segment].reset_index(drop=True)

            last_start = len(segment) - LOOKBACK - HORIZON
            for start in range(0, last_start + 1, EXPORT_EVERY_STEPS):
                hist_scaled = values_scaled[start:start + LOOKBACK]
                hist_raw = values_raw[start:start + LOOKBACK]

                x = torch.from_numpy(hist_scaled).unsqueeze(0).to(device)
                pred_scaled = forecast_model(x)
                if isinstance(pred_scaled, (tuple, list)):
                    pred_scaled = pred_scaled[0]

                pred_scaled = pred_scaled.squeeze(0).cpu().numpy()
                pred_raw = scaler.inverse_transform(pred_scaled)

                # Risk features must match training: do NOT clip predictions here.
                risk_features = make_risk_features(hist_raw, pred_raw)
                risk_probability = float(
                    risk_model.predict_proba(
                        risk_features.reshape(1, -1)
                    )[0, 1]
                )

                # Physical demand forecasts should not be negative.
                demand_forecast = np.maximum(pred_raw, 0.0)
                current = hist_raw[-1]
                decision_time = segment_ts.iloc[start + LOOKBACK - 1]

                row = {
                    "vm_id": vm_id,
                    "decision_time": decision_time,
                    "forecast_start": decision_time + pd.Timedelta(minutes=5),
                    "forecast_end": decision_time + pd.Timedelta(minutes=60),
                    "current_cpu_pct": current[0],
                    "current_memory_kb": current[1],
                    "current_disk_read_kbps": current[2],
                    "current_disk_write_kbps": current[3],
                    "risk_probability": risk_probability,
                    "risk_threshold": float(risk_threshold),
                    "risk_alert": int(risk_probability >= risk_threshold),
                }

                names = ["cpu", "memory", "disk_read", "disk_write"]
                for j, name in enumerate(names):
                    row[f"forecast_{name}_max"] = float(
                        demand_forecast[:, j].max()
                    )
                    row[f"forecast_{name}_mean"] = float(
                        demand_forecast[:, j].mean()
                    )

                records.append(row)

        if file_idx % 200 == 0:
            print(f"Scanned {file_idx}/{len(files)} VM test files")

if not records:
    raise RuntimeError(
        "No forecasts exported. Check test files, timestamps and model inputs."
    )

export_df = pd.DataFrame(records)
export_df["decision_time"] = pd.to_datetime(export_df["decision_time"])
export_df["forecast_start"] = pd.to_datetime(export_df["forecast_start"])
export_df["forecast_end"] = pd.to_datetime(export_df["forecast_end"])

export_df = export_df.sort_values(
    ["decision_time", "vm_id"]
).reset_index(drop=True)

output_file = OUT_DIR / "bitbrains_cloudsim_predictions.csv"
export_df.to_csv(output_file, index=False)

print("\nCLOUDSIM EXPORT COMPLETE")
print("Rows:", len(export_df))
print("Unique VMs:", export_df["vm_id"].nunique())
print("Decision interval:", STEP_MINUTES, "minutes")
print("Risk alerts:", int(export_df["risk_alert"].sum()))
print("Output:", output_file)
print("\nPreview:")
display(export_df.head())


Scanned 200/1217 VM test files
Scanned 400/1217 VM test files
Scanned 600/1217 VM test files
Scanned 800/1217 VM test files
Scanned 1000/1217 VM test files
Scanned 1200/1217 VM test files

CLOUDSIM EXPORT COMPLETE
Rows: 456799
Unique VMs: 1217
Decision interval: 15 minutes
Risk alerts: 47996
Output: /content/drive/MyDrive/Bitbrains_Fresh_Research/results/cloudsim/bitbrains_cloudsim_predictions.csv

Preview:


,vm_id,decision_time,forecast_start,forecast_end,current_cpu_pct,current_memory_kb,current_disk_read_kbps,current_disk_write_kbps,risk_probability,risk_threshold,risk_alert,forecast_cpu_max,forecast_cpu_mean,forecast_memory_max,forecast_memory_mean,forecast_disk_read_max,forecast_disk_read_mean,forecast_disk_write_max,forecast_disk_write_mean
0,694,2013-08-20 09:15:00,2013-08-20 09:20:00,2013-08-20 10:15:00,2.666667,444594.400000,0.000000,24.400000,0.178813,0.66,0,3.141787,2.625566,365123.875000,322247.718750,124.067360,20.448919,42.335609,17.433386
1,695,2013-08-20 09:25:00,2013-08-20 09:30:00,2013-08-20 10:25:00,1.600000,167772.000000,0.000000,17.600000,0.128816,0.66,0,2.116544,1.807517,187171.578125,147474.375000,78.607414,8.444929,32.996708,11.154477
2,694,2013-08-20 09:30:00,2013-08-20 09:35:00,2013-08-20 10:30:00,2.200000,184547.200000,0.733333,21.000000,0.177981,0.66,0,2.840039,2.413025,353342.843750,278818.875000,87.607674,10.633981,38.673222,14.878807
3,695,2013-08-20 09:40:00,2013-08-20 09:45:00,2013-08-20 10:40:00,1.500000,156586.933333,0.000000,14.733333,0.132390,0.66,0,2.177544,1.810284,207411.328125,164233.984375,78.607414,8.444929,34.229298,11.445619
4,694,2013-08-20 09:45:00,2013-08-20 09:50:00,2013-08-20 10:45:00,2.133333,268432.800000,0.000000,18.933333,0.164017,0.66,0,2.856145,2.382184,306824.625000,282276.468750,101.432884,12.197957,39.011139,14.762818
